# ЛР № 3. Потоковая обработка банковских транзакций: Apache Kafka (Aiven) + Spark Structured Streaming

**Вариант № 2 — банковские транзакции.** Поля: `transaction_id`, `client_id`, `transaction_type`, `amount`, `currency`, `city`, `device_type`, `timestamp` (+ дополнительное `fee` — комиссия).

**Поток данных:** генератор → `raw_data.csv` в Google Drive → пакетная обработка PySpark → продюсер отправляет новые транзакции в топик Aiven Kafka по SSL → Spark Structured Streaming читает топик, проверяет сообщения и сохраняет принятые → итоговые показатели (история + поток) пересчитываются после каждого микропакета → результаты, промежуточные данные и журнал сохраняются в Google Drive.

**Перед запуском** (подробно — README.md, раздел «Aiven»):
1. Сервис Aiven for Apache Kafka в состоянии **Running**, создан топик `bank-transactions-v2`.
2. Скачаны `ca.pem`, `service.cert`, `service.key` (Quick connect → Python). Их можно сохранить в «Секретах» Colab (значок ключа слева) или загрузить в ячейке раздела 7. **Не кладите их в Google Drive.**
3. В «Секретах» Colab задан `KAFKA_BOOTSTRAP_SERVERS` (Service URI вида `host:port`) с доступом для этого блокнота.

**Порядок:** выполняйте ячейки сверху вниз (Runtime → Run all). Повторный запуск ячеек безопасен: SparkSession не создаётся заново, потоки перед новым запуском останавливаются, генерация пропускается, если данные уже есть.

Среда — **Google Colab** (одна ВМ, Spark в режиме `local[*]`), а не Google Cloud Dataproc и не распределённый кластер.

## 0. Подготовка среды

In [ ]:
# Определение среды, установка зависимостей, проверка Java/OpenSSL/keytool (повторный запуск безопасен)
import importlib, os, shutil, subprocess, sys, tempfile

IN_COLAB = "google.colab" in sys.modules
CODE_DIR = "/content/lab3_code" if IN_COLAB else os.environ.get("LAB3_CODE_DIR", tempfile.mkdtemp(prefix="lab3_code_"))
os.makedirs(CODE_DIR, exist_ok=True)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

try:
    import confluent_kafka  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "confluent-kafka==2.16.0"], check=True)
import confluent_kafka
import pyspark

def _ver(cmd, must_contain=""):
    p = subprocess.run(cmd, capture_output=True, text=True)
    lines = (p.stdout + p.stderr).strip().splitlines()
    return next((l for l in lines if must_contain in l), lines[0] if lines else "?")

java_line = _ver(["java", "-version"], 'version "')
print("Среда            :", "Google Colab" if IN_COLAB else "локально (отладка)")
print("Python           :", sys.version.split()[0])
print("PySpark          :", pyspark.__version__, "(проверено с 4.0.4)")
print("Java             :", java_line)
_ck = confluent_kafka.version()
print("confluent-kafka  :", _ck if isinstance(_ck, str) else _ck[0], "/ librdkafka", confluent_kafka.libversion()[0])
print("OpenSSL          :", _ver(["openssl", "version"]))
print("keytool          :", shutil.which("keytool") or "НЕ НАЙДЕН")
import re
m = re.search(r'version "(\d+)', java_line)
if not m or int(m.group(1)) < 17:
    raise RuntimeError("PySpark 4 требует Java 17+. В Colab по умолчанию установлен OpenJDK 21 — проверьте вывод java -version.")

## 1. Код проекта

Ячейки ниже записывают модули проекта в `/content/lab3_code` (локальный диск ВМ). Это те же файлы, что лежат в репозитории; блокнот собирается из них скриптом `tools/build_notebook.py`. Ячейки можно свернуть — достаточно их выполнить.

In [ ]:
%%writefile {CODE_DIR}/lab3_common.py
"""Общие настройки, справочники и вспомогательные функции ЛР № 3 (вариант № 2).

Модуль не зависит от Spark и Kafka: его импортируют генератор, продюсер,
Spark-задания, тесты и блокнот Colab. Все настраиваемые параметры собраны в
dataclass Settings; значения по умолчанию можно переопределить переменными
окружения с префиксом LAB3_ (см. .env.example).
"""
from __future__ import annotations

import csv
import json
import logging
import os
import sys
import tempfile
from dataclasses import asdict, dataclass, field
from pathlib import Path

# ---------------------------------------------------------------------------
# Схема варианта № 2
# ---------------------------------------------------------------------------
REQUIRED_FIELDS = [
    "transaction_id", "client_id", "transaction_type", "amount",
    "currency", "city", "device_type", "timestamp",
]
EXTRA_FIELDS = ["fee"]  # комиссия банка: второй осмысленный числовой признак
ALL_FIELDS = REQUIRED_FIELDS + EXTRA_FIELDS

TRANSACTION_TYPES = {"payment": 0.42, "transfer": 0.24, "withdrawal": 0.16, "deposit": 0.12, "refund": 0.06}
CURRENCIES = {"RUB": 0.62, "USD": 0.15, "EUR": 0.14, "CNY": 0.09}
CITIES = {
    "Moscow": 0.30, "Saint Petersburg": 0.16, "Novosibirsk": 0.08, "Yekaterinburg": 0.08,
    "Kazan": 0.07, "Nizhny Novgorod": 0.06, "Krasnodar": 0.06, "Samara": 0.05,
    "Rostov-on-Don": 0.05, "Vladivostok": 0.04, "Kaliningrad": 0.03, "Tomsk": 0.02,
}
DEVICE_TYPES = {"mobile": 0.52, "desktop": 0.21, "atm": 0.12, "pos_terminal": 0.10, "tablet": 0.05}

# Условные фиксированные курсы для приведения сумм к одной валюте (EUR).
# Не являются рыночными котировками; нужны, чтобы не складывать рубли с долларами.
EUR_RATES = {"RUB": "0.0105", "USD": "0.92", "EUR": "1.00", "CNY": "0.13"}
MAX_REASONABLE_AMOUNT = 10_000_000  # сумма больше — явная ошибка ввода
TIMESTAMP_FORMAT = "%Y-%m-%d %H:%M:%S"  # формат в данных
SPARK_TIMESTAMP_FORMAT = "yyyy-MM-dd HH:mm:ss"
UNKNOWN = "unknown"
UNKNOWN_CITY = "Unknown"


# ---------------------------------------------------------------------------
# Настройки
# ---------------------------------------------------------------------------
def _env(name: str, default):
    raw = os.environ.get(f"LAB3_{name}")
    if raw in (None, ""):
        return default
    if isinstance(default, bool):
        return raw.strip().lower() in ("1", "true", "yes", "on")
    return type(default)(raw)


@dataclass
class Settings:
    # Корневой каталог проекта в Google Drive (или локально при отладке)
    base_dir: str = _env("BASE_DIR", "/content/drive/MyDrive/lab3_bank")
    # Локальный рабочий каталог Spark (checkpoint, хранилище потоковых транзакций).
    # Держится на диске ВМ Colab и синхронизируется в Drive (см. README, раздел «Хранение»).
    work_dir: str = _env("WORK_DIR", "/content/lab3_work")
    secrets_dir: str = _env("SECRETS_DIR", "/content/kafka_secrets")  # НЕ в Google Drive

    # Генерация данных
    seed: int = _env("SEED", 42)
    raw_rows: int = _env("RAW_ROWS", 500_000)
    new_rows: int = _env("NEW_ROWS", 300)
    new_seed: int = _env("NEW_SEED", 2026)
    batch_tag: str = _env("BATCH_TAG", "B001")
    period_start: str = _env("PERIOD_START", "2025-01-01")
    period_end: str = _env("PERIOD_END", "2026-01-01")  # не включается
    new_period_start: str = _env("NEW_PERIOD_START", "2026-01-01")

    # Kafka (Aiven)
    topic: str = _env("TOPIC", "bank-transactions-v2")
    auth_mode: str = _env("AUTH_MODE", "certificate")  # certificate | sasl
    sasl_mechanism: str = _env("SASL_MECHANISM", "SCRAM-SHA-256")
    consumer_group: str = _env("CONSUMER_GROUP", "lab3-verify")

    # Spark Structured Streaming
    trigger_seconds: int = _env("TRIGGER_SECONDS", 5)
    starting_offsets: str = _env("STARTING_OFFSETS", "earliest")
    max_offsets_per_trigger: int = _env("MAX_OFFSETS_PER_TRIGGER", 5000)
    shuffle_partitions: int = _env("SHUFFLE_PARTITIONS", 4)
    kafka_package: str = _env("KAFKA_PACKAGE", "")  # пусто = по версии PySpark

    extra: dict = field(default_factory=dict)

    # ---- производные пути
    @property
    def data_dir(self) -> Path:
        return Path(self.base_dir) / "lab3_data"

    @property
    def results_dir(self) -> Path:
        return Path(self.base_dir) / "lab3_results"

    @property
    def intermediate_dir(self) -> Path:
        """Промежуточные данные в Drive: очищенная история и копия рабочего каталога."""
        return Path(self.base_dir) / "lab3_intermediate"

    @property
    def raw_csv(self) -> Path:
        return self.data_dir / "raw_data.csv"

    @property
    def new_jsonl(self) -> Path:
        return self.data_dir / "new_transactions.jsonl"

    @property
    def new_csv(self) -> Path:
        return self.data_dir / "new_transactions.csv"

    @property
    def clean_history_dir(self) -> Path:
        return Path(self.work_dir) / "clean_history.parquet"

    @property
    def stream_store_dir(self) -> Path:
        return Path(self.work_dir) / "stream_transactions"

    @property
    def stream_rejected_dir(self) -> Path:
        return Path(self.work_dir) / "stream_rejected"

    @property
    def checkpoint_dir(self) -> Path:
        return Path(self.work_dir) / "checkpoints"

    @property
    def work_backup_dir(self) -> Path:
        return self.intermediate_dir / "work_backup"

    def ensure_dirs(self):
        for p in (self.data_dir, self.results_dir, self.intermediate_dir,
                  Path(self.work_dir), self.checkpoint_dir):
            p.mkdir(parents=True, exist_ok=True)

    def as_dict(self):
        d = asdict(self)
        d.pop("extra", None)
        return d


# ---------------------------------------------------------------------------
# Журнал выполнения
# ---------------------------------------------------------------------------
def setup_logging(log_file: Path | str | None, level=logging.INFO) -> logging.Logger:
    """Логгер lab3: консоль + файл (дозапись). Повторный вызов не дублирует обработчики."""
    logger = logging.getLogger("lab3")
    logger.setLevel(level)
    logger.propagate = False
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s", "%Y-%m-%d %H:%M:%S")
    have_console = any(getattr(h, "_lab3", None) == "console" for h in logger.handlers)
    if not have_console:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(fmt)
        h._lab3 = "console"
        logger.addHandler(h)
    if log_file:
        log_file = str(log_file)
        for h in list(logger.handlers):
            if getattr(h, "_lab3", None) == "file" and h.baseFilename != os.path.abspath(log_file):
                logger.removeHandler(h)
                h.close()
        if not any(getattr(h, "_lab3", None) == "file" for h in logger.handlers):
            Path(log_file).parent.mkdir(parents=True, exist_ok=True)
            fh = logging.FileHandler(log_file, encoding="utf-8")
            fh.setFormatter(fmt)
            fh._lab3 = "file"
            logger.addHandler(fh)
    return logger


# ---------------------------------------------------------------------------
# Файлы результатов
# ---------------------------------------------------------------------------
def write_csv_atomic(path: Path | str, rows: list[dict], fieldnames: list[str] | None = None):
    """Записать небольшой CSV атомарно: временный файл + os.replace.

    Повторный запуск перезаписывает файл целиком, а читатель никогда не увидит
    наполовину записанный файл.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else []
    fd, tmp = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=str(path.parent))
    try:
        with os.fdopen(fd, "w", newline="", encoding="utf-8") as fh:
            w = csv.DictWriter(fh, fieldnames=fieldnames, extrasaction="ignore")
            w.writeheader()
            w.writerows(rows)
        os.chmod(tmp, 0o644)  # mkstemp создаёт файл с правами 600
        os.replace(tmp, path)
    except BaseException:
        if os.path.exists(tmp):
            os.unlink(tmp)
        raise


def read_csv(path: Path | str) -> list[dict]:
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def write_json(path: Path | str, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=str(path.parent))
    with os.fdopen(fd, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, ensure_ascii=False, indent=2, default=str)
    os.chmod(tmp, 0o644)
    os.replace(tmp, path)


def compare_dimension_stats(before: list[dict], after: list[dict]) -> list[dict]:
    """Таблица изменений «до/после» по ключу (dimension, value)."""
    def key(r):
        return r["dimension"], r["value"]

    b = {key(r): r for r in before}
    a = {key(r): r for r in after}
    out = []
    for k in sorted(set(b) | set(a)):
        rb, ra = b.get(k, {}), a.get(k, {})
        cb, ca = int(rb.get("tx_count", 0) or 0), int(ra.get("tx_count", 0) or 0)
        sb, sa = float(rb.get("amount_eur_sum", 0) or 0), float(ra.get("amount_eur_sum", 0) or 0)
        out.append({
            "dimension": k[0], "value": k[1],
            "tx_count_before": cb, "tx_count_after": ca, "tx_count_delta": ca - cb,
            "amount_eur_sum_before": round(sb, 2), "amount_eur_sum_after": round(sa, 2),
            "amount_eur_sum_delta": round(sa - sb, 2),
            "avg_amount_eur_before": round(sb / cb, 2) if cb else None,
            "avg_amount_eur_after": round(sa / ca, 2) if ca else None,
        })
    return out


def show_rows(rows: list[dict], limit: int = 20, title: str | None = None):
    """Показать список словарей таблицей (pandas в Colab, иначе текстом)."""
    if title:
        print(title)
    rows = rows[:limit]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(rows))
    except Exception:
        if not rows:
            print("(нет строк)")
            return
        cols = list(rows[0])
        print(" | ".join(cols))
        for r in rows:
            print(" | ".join(str(r.get(c, "")) for c in cols))

In [ ]:
%%writefile {CODE_DIR}/generate_data.py
#!/usr/bin/env python3
"""Генератор синтетических банковских транзакций (ЛР № 3, вариант № 2).

Два режима:
  raw — основной исторический набор (CSV, ~500 000 строк) с пропусками,
        дубликатами и некорректными значениями;
  new — небольшой пакет новых транзакций для отправки в Kafka (JSON Lines + CSV),
        включает несколько некорректных сообщений, повторы и битый JSON.

Данные полностью синтетические, генерация воспроизводима (seed).
Нужна только стандартная библиотека Python 3.10+.

  python generate_data.py raw --rows 500000 --out data/raw_data.csv
  python generate_data.py new --rows 300 --out data/new_transactions.jsonl
"""
from __future__ import annotations

import argparse
import csv
import json
import math
import random
import time
from collections import Counter
from datetime import datetime, timedelta
from pathlib import Path

import lab3_common as C

RUB_PER_UNIT = {"RUB": 1.0, "USD": 88.0, "EUR": 95.0, "CNY": 12.2}  # условные
TYPE_PROFILE = {  # медиана суммы в рублях, разброс логнормального распределения, ставка комиссии
    "payment": (1_500, 1.0, 0.0), "transfer": (8_000, 1.1, 0.005), "withdrawal": (5_000, 0.8, 0.01),
    "deposit": (20_000, 1.0, 0.0), "refund": (1_200, 0.9, 0.0),
}
CITY_LEVEL = {"Moscow": 1.30, "Saint Petersburg": 1.15, "Kaliningrad": 0.95, "Vladivostok": 1.05}
HOUR_WEIGHTS = [1, 1, 1, 1, 1, 2, 4, 7, 9, 10, 10, 10, 11, 11, 10, 10, 10, 11, 12, 12, 10, 7, 4, 2]

INVALID_KINDS = [
    "negative_amount", "zero_amount", "non_numeric_amount", "huge_amount", "unknown_currency",
    "unknown_city", "unknown_device", "unknown_type", "malformed_timestamp", "timestamp_out_of_range",
]
FIXABLE_KINDS = ["case_and_spaces", "lowercase_currency", "iso_timestamp"]


def pick(rng: random.Random, weights: dict) -> str:
    keys = list(weights)
    return rng.choices(keys, weights=[weights[k] for k in keys], k=1)[0]


class TransactionFactory:
    """Порождает корректные транзакции; дефекты вносятся отдельно."""

    def __init__(self, rng: random.Random, n_clients: int, start: datetime, span_seconds: int):
        self.rng, self.start, self.span = rng, start, span_seconds
        cities = list(C.CITIES)
        self.clients = []
        for i in range(1, n_clients + 1):
            home = rng.choices(cities, weights=[C.CITIES[c] for c in cities], k=1)[0]
            currency = pick(rng, C.CURRENCIES)
            device = pick(rng, C.DEVICE_TYPES)
            self.clients.append((f"C{i:07d}", home, currency, device))
        self.day_count = max(1, span_seconds // 86_400)

    def make(self, transaction_id: str) -> dict:
        rng = self.rng
        client_id, home, currency, device = self.clients[rng.randrange(len(self.clients))]
        city = home if rng.random() < 0.9 else pick(rng, C.CITIES)
        if rng.random() < 0.15:
            currency = pick(rng, C.CURRENCIES)
        if rng.random() < 0.2:
            device = pick(rng, C.DEVICE_TYPES)
        tx_type = pick(rng, C.TRANSACTION_TYPES)
        if tx_type == "withdrawal" and rng.random() < 0.7:
            device = "atm"
        median, sigma, fee_rate = TYPE_PROFILE[tx_type]
        rub = rng.lognormvariate(math.log(median * CITY_LEVEL.get(city, 1.0)), sigma)
        amount = round(max(rub / RUB_PER_UNIT[currency], 0.5), 2)
        fee = round(amount * fee_rate, 2)
        day = rng.randrange(self.day_count)
        hour = rng.choices(range(24), weights=HOUR_WEIGHTS, k=1)[0]
        ts = self.start + timedelta(days=day, hours=hour, seconds=rng.randrange(3600))
        return {
            "transaction_id": transaction_id, "client_id": client_id, "transaction_type": tx_type,
            "amount": f"{amount:.2f}", "currency": currency, "city": city, "device_type": device,
            "timestamp": ts.strftime(C.TIMESTAMP_FORMAT), "fee": f"{fee:.2f}",
        }


def apply_invalid(rng, rec, kind):
    if kind == "negative_amount":
        rec["amount"] = "-" + rec["amount"]
    elif kind == "zero_amount":
        rec["amount"] = "0.00"
    elif kind == "non_numeric_amount":
        rec["amount"] = rng.choice(["abc", "1,250.00", "N/A", "12.5.1"])
    elif kind == "huge_amount":
        rec["amount"] = f"{rng.uniform(5e7, 9e8):.2f}"
    elif kind == "unknown_currency":
        rec["currency"] = rng.choice(["XXX", "RUR", "$$"])
    elif kind == "unknown_city":
        rec["city"] = rng.choice(["Atlantis", "N/A", "Gotham"])
    elif kind == "unknown_device":
        rec["device_type"] = rng.choice(["smart_fridge", "fax"])
    elif kind == "unknown_type":
        rec["transaction_type"] = rng.choice(["chargeback_x", "???"])
    elif kind == "malformed_timestamp":
        rec["timestamp"] = rng.choice(["2025-02-30 12:00:00", "31/12/2025 10:00", "not_a_date", "2025-13-01 25:00:00"])
    elif kind == "timestamp_out_of_range":
        rec["timestamp"] = rng.choice(["2031-01-01 00:00:00", "1999-12-31 23:59:59"])


def apply_fixable(rng, rec, kind):
    if kind == "case_and_spaces":
        rec["city"] = rng.choice([rec["city"].upper(), f"  {rec['city']} ", rec["city"].lower()])
        rec["device_type"] = rec["device_type"].upper()
    elif kind == "lowercase_currency":
        rec["currency"] = rec["currency"].lower()
    elif kind == "iso_timestamp":
        rec["timestamp"] = rec["timestamp"].replace(" ", "T")


def partial_duplicate(rng, rec):
    """Повтор той же транзакции с иным оформлением (регистр города, формат даты)."""
    dup = dict(rec)
    if dup["city"].strip():
        dup["city"] = dup["city"].strip().upper()
    elif " " in dup["timestamp"]:
        dup["timestamp"] = dup["timestamp"].replace(" ", "T")
    return dup if dup != rec else None


def generate_raw(out: Path, rows: int, seed: int, start: str, end: str,
                 missing_rate=0.02, invalid_rate=0.01, fixable_rate=0.02, duplicate_rate=0.01,
                 partial_share=0.3) -> dict:
    """Основной набор. Возвращает и сохраняет статистику генерации (<out>.stats.json)."""
    rng = random.Random(seed)
    t0 = time.time()
    s, e = datetime.fromisoformat(start), datetime.fromisoformat(end)
    factory = TransactionFactory(rng, max(100, rows // 25), s, int((e - s).total_seconds()))
    stats, missing, invalid, fixable = Counter(), Counter(), Counter(), Counter()
    pool = []
    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open("w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=C.ALL_FIELDS)
        w.writeheader()
        for i in range(1, rows + 1):
            rec = factory.make(f"T{i:09d}")
            if rng.random() < fixable_rate:
                k = rng.choice(FIXABLE_KINDS)
                apply_fixable(rng, rec, k)
                fixable[k] += 1
            if rng.random() < invalid_rate:
                k = rng.choice(INVALID_KINDS)
                apply_invalid(rng, rec, k)
                invalid[k] += 1
            if rng.random() < missing_rate:
                f = rng.choice(C.REQUIRED_FIELDS)
                rec[f] = rng.choice(["", " "])
                missing[f] += 1
            w.writerow(rec)
            stats["base_rows"] += 1
            if len(pool) < 20_000:
                pool.append(rec)
            elif rng.random() < 20_000 / i:
                pool[rng.randrange(20_000)] = rec
        for _ in range(int(round(rows * duplicate_rate))):
            src = pool[rng.randrange(len(pool))]
            dup = partial_duplicate(rng, src) if rng.random() < partial_share else None
            if dup is not None:
                w.writerow(dup)
                stats["partial_duplicates"] += 1
            else:
                w.writerow(src)
                stats["exact_duplicates"] += 1
    result = {
        "file": str(out), "seed": seed, "period": [start, end], "fields": C.ALL_FIELDS,
        "base_rows": stats["base_rows"], "exact_duplicates_added": stats["exact_duplicates"],
        "partial_duplicates_added": stats["partial_duplicates"],
        "total_rows_in_file": stats["base_rows"] + stats["exact_duplicates"] + stats["partial_duplicates"],
        "rows_with_missing_value": sum(missing.values()), "missing_by_field": dict(sorted(missing.items())),
        "rows_with_invalid_value": sum(invalid.values()), "invalid_by_kind": dict(sorted(invalid.items())),
        "rows_with_fixable_deviation": sum(fixable.values()), "fixable_by_kind": dict(sorted(fixable.items())),
        "rates": {"missing": missing_rate, "invalid": invalid_rate, "fixable": fixable_rate,
                  "duplicate": duplicate_rate, "partial_share": partial_share},
        "file_size_mb": round(out.stat().st_size / 2 ** 20, 2),
        "seconds": round(time.time() - t0, 2),
    }
    C.write_json(out.with_suffix(".stats.json"), result)
    return result


def generate_new(out_jsonl: Path, rows: int, seed: int, start: str, batch_tag: str,
                 invalid=5, repeats=3, malformed=2, out_csv: Path | None = None) -> dict:
    """Пакет новых транзакций для Kafka.

    В JSON Lines попадают: `rows` корректных транзакций, `invalid` некорректных
    (неверная сумма, пропущенное обязательное поле, неизвестная валюта, плохая дата),
    `repeats` повторов уже включённых транзакций (тот же transaction_id) и
    `malformed` строк с битым JSON. Порядок перемешивается детерминированно.
    """
    rng = random.Random(seed)
    s = datetime.fromisoformat(start)
    factory = TransactionFactory(rng, max(50, rows // 3), s, 86_400)
    good = [factory.make(f"S-{batch_tag}-{i:06d}") for i in range(1, rows + 1)]
    for r in good:  # в JSON суммы — числа, как у реального источника
        r["amount"], r["fee"] = float(r["amount"]), float(r["fee"])
    bad = []
    kinds = ["negative_amount", "missing_client_id", "unknown_currency", "bad_timestamp", "amount_as_text"]
    for j in range(invalid):
        r = factory.make(f"S-{batch_tag}-X{j + 1:04d}")
        r["amount"], r["fee"] = float(r["amount"]), float(r["fee"])
        kind = kinds[j % len(kinds)]
        if kind == "negative_amount":
            r["amount"] = -abs(r["amount"])
        elif kind == "missing_client_id":
            del r["client_id"]
        elif kind == "unknown_currency":
            r["currency"] = "XXX"
        elif kind == "bad_timestamp":
            r["timestamp"] = "2026-02-30 25:61:00"
        else:
            r["amount"] = "abc"
        bad.append(r)
    dups = [dict(good[rng.randrange(len(good))]) for _ in range(min(repeats, len(good)))]
    lines = [json.dumps(r, ensure_ascii=False) for r in good + bad + dups]
    lines += ['{"transaction_id": "S-%s-BROKEN-%d", "amount": ' % (batch_tag, k + 1) for k in range(malformed)]
    rng.shuffle(lines)
    out_jsonl.parent.mkdir(parents=True, exist_ok=True)
    out_jsonl.write_text("\n".join(lines) + "\n", encoding="utf-8")
    if out_csv is not None:
        with out_csv.open("w", newline="", encoding="utf-8") as fh:
            w = csv.DictWriter(fh, fieldnames=C.ALL_FIELDS, extrasaction="ignore")
            w.writeheader()
            for line in lines:
                try:
                    w.writerow(json.loads(line))
                except json.JSONDecodeError:
                    continue
    result = {"file": str(out_jsonl), "seed": seed, "batch_tag": batch_tag, "valid_transactions": rows,
              "invalid_transactions": invalid, "repeated_messages": len(dups),
              "malformed_json_lines": malformed, "total_messages": len(lines),
              "expected_accepted": rows, "start": start}
    C.write_json(out_jsonl.with_suffix(".stats.json"), result)
    return result


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="mode", required=True)
    s = C.Settings()
    r = sub.add_parser("raw", help="основной набор CSV")
    r.add_argument("--rows", type=int, default=s.raw_rows)
    r.add_argument("--seed", type=int, default=s.seed)
    r.add_argument("--out", type=Path, default=Path("data/raw_data.csv"))
    r.add_argument("--start", default=s.period_start)
    r.add_argument("--end", default=s.period_end)
    r.add_argument("--missing-rate", type=float, default=0.02)
    r.add_argument("--invalid-rate", type=float, default=0.01)
    r.add_argument("--duplicate-rate", type=float, default=0.01)
    n = sub.add_parser("new", help="пакет новых транзакций для Kafka (JSON Lines)")
    n.add_argument("--rows", type=int, default=s.new_rows)
    n.add_argument("--seed", type=int, default=s.new_seed)
    n.add_argument("--out", type=Path, default=Path("data/new_transactions.jsonl"))
    n.add_argument("--start", default=s.new_period_start)
    n.add_argument("--batch-tag", default=s.batch_tag)
    n.add_argument("--invalid", type=int, default=5)
    n.add_argument("--repeats", type=int, default=3)
    n.add_argument("--malformed", type=int, default=2)
    a = p.parse_args()
    if a.mode == "raw":
        res = generate_raw(a.out, a.rows, a.seed, a.start, a.end, a.missing_rate, a.invalid_rate,
                           duplicate_rate=a.duplicate_rate)
    else:
        res = generate_new(a.out, a.rows, a.seed, a.start, a.batch_tag, a.invalid, a.repeats,
                           a.malformed, out_csv=a.out.with_suffix(".csv"))
    print(json.dumps(res, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/kafka_security.py
"""SSL/SASL-подключение к Aiven for Apache Kafka для Python-клиента и Spark (JVM).

Способы аутентификации Aiven:
  * certificate — взаимный TLS (mTLS): ca.pem + service.cert + service.key;
  * sasl        — SASL_SSL с SCRAM-SHA-256 (логин/пароль) + ca.pem.

Python-клиент (confluent-kafka / librdkafka) читает PEM-файлы напрямую.
Spark Kafka connector работает в JVM (kafka-clients 3.9.1). Для неё из PEM
создаются:
  * truststore (PKCS12) с CA-сертификатом — keytool;
  * keystore (PKCS12) с клиентским сертификатом и приватным ключом — openssl.
Пароли хранилищ генерируются случайно на сессию и передаются утилитам через
переменные окружения, а не в командной строке. Содержимое ключей не выводится.

Секреты кладутся только в secrets_dir на диске ВМ (по умолчанию
/content/kafka_secrets), но не в Google Drive.
"""
from __future__ import annotations

import os
import re
import secrets as pysecrets
import shutil
import stat
import subprocess
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

CA_FILE, CERT_FILE, KEY_FILE = "ca.pem", "service.cert", "service.key"
TRUSTSTORE, KEYSTORE = "client.truststore.p12", "client.keystore.p12"


class KafkaSecurityError(RuntimeError):
    """Понятная ошибка настройки безопасности (без секретов в тексте)."""


@dataclass
class KafkaConnection:
    bootstrap_servers: str
    auth_mode: str = "certificate"          # certificate | sasl
    secrets_dir: Path = Path("/content/kafka_secrets")
    sasl_username: str | None = None
    sasl_password: str | None = field(default=None, repr=False)
    sasl_mechanism: str = "SCRAM-SHA-256"
    store_password: str = field(default_factory=lambda: pysecrets.token_urlsafe(24), repr=False)

    # ---- пути
    @property
    def ca(self) -> Path:
        return Path(self.secrets_dir) / CA_FILE

    @property
    def cert(self) -> Path:
        return Path(self.secrets_dir) / CERT_FILE

    @property
    def key(self) -> Path:
        return Path(self.secrets_dir) / KEY_FILE

    @property
    def truststore(self) -> Path:
        return Path(self.secrets_dir) / TRUSTSTORE

    @property
    def keystore(self) -> Path:
        return Path(self.secrets_dir) / KEYSTORE

    def __post_init__(self):
        self.secrets_dir = Path(self.secrets_dir)
        self.bootstrap_servers = (self.bootstrap_servers or "").strip()
        if self.auth_mode not in ("certificate", "sasl"):
            raise KafkaSecurityError(f"auth_mode должен быть certificate или sasl, получено {self.auth_mode!r}")
        if not re.fullmatch(r"[A-Za-z0-9.\-]+:\d{1,5}(,[A-Za-z0-9.\-]+:\d{1,5})*", self.bootstrap_servers):
            raise KafkaSecurityError(
                "bootstrap_servers должен иметь вид host:port (Aiven: <service>-<project>.<...>.aivencloud.com:<port>)")

    # ---- конфигурации клиентов
    def confluent_config(self, **extra) -> dict:
        """Параметры для confluent_kafka.Producer / Consumer / AdminClient."""
        conf = {"bootstrap.servers": self.bootstrap_servers, "ssl.ca.location": str(self.ca),
                "client.id": "lab3-colab"}
        if self.auth_mode == "certificate":
            conf.update({"security.protocol": "SSL",
                         "ssl.certificate.location": str(self.cert),
                         "ssl.key.location": str(self.key)})
        else:
            conf.update({"security.protocol": "SASL_SSL", "sasl.mechanism": self.sasl_mechanism,
                         "sasl.username": self.sasl_username, "sasl.password": self.sasl_password})
        conf.update(extra)
        return conf

    def spark_options(self) -> dict:
        """Опции для spark.readStream.format("kafka"): префикс kafka. передаёт их kafka-clients."""
        opts = {
            "kafka.bootstrap.servers": self.bootstrap_servers,
            "kafka.ssl.truststore.location": str(self.truststore),
            "kafka.ssl.truststore.type": "PKCS12",
            "kafka.ssl.truststore.password": self.store_password,
            # проверка имени хоста включена по умолчанию в kafka-clients >= 2.0; задаём явно
            "kafka.ssl.endpoint.identification.algorithm": "https",
        }
        if self.auth_mode == "certificate":
            opts.update({
                "kafka.security.protocol": "SSL",
                "kafka.ssl.keystore.location": str(self.keystore),
                "kafka.ssl.keystore.type": "PKCS12",
                "kafka.ssl.keystore.password": self.store_password,
                "kafka.ssl.key.password": self.store_password,
            })
        else:
            user = self.sasl_username.replace('"', '\\"')
            pwd = self.sasl_password.replace('"', '\\"')
            opts.update({
                "kafka.security.protocol": "SASL_SSL",
                "kafka.sasl.mechanism": self.sasl_mechanism,
                "kafka.sasl.jaas.config":
                    f'org.apache.kafka.common.security.scram.ScramLoginModule required username="{user}" password="{pwd}";',
            })
        return opts

    def describe(self) -> dict:
        """Безопасное описание конфигурации для журнала (без паролей и ключей)."""
        host = self.bootstrap_servers.split(",")[0]
        masked = re.sub(r"^[^.]+", lambda m: m.group(0)[:4] + "***", host)
        return {"bootstrap": masked, "auth_mode": self.auth_mode, "secrets_dir": str(self.secrets_dir),
                "truststore": self.truststore.name, "keystore": self.keystore.name if self.auth_mode == "certificate" else None}


# ---------------------------------------------------------------------------
# Размещение секретов
# ---------------------------------------------------------------------------
def prepare_secrets_dir(path: Path | str) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    os.chmod(path, stat.S_IRWXU)  # 700
    if "/drive/" in str(path).lower():
        raise KafkaSecurityError("Каталог секретов не должен находиться в Google Drive")
    return path


_PEM_BLOCK = re.compile(r"-----BEGIN ([A-Z0-9 ]+)-----(.*?)-----END \1-----", re.S)


def normalize_pem(text: str) -> str:
    """Восстановить переносы строк PEM, если значение вставлено одной строкой
    (например, в поле «Секреты» Colab). Корректный PEM не меняется."""
    blocks = _PEM_BLOCK.findall(text)
    if not blocks:
        return text
    out = []
    for label, body in blocks:
        b64 = re.sub(r"\s+", "", body)
        lines = [b64[i:i + 64] for i in range(0, len(b64), 64)]
        out.append("\n".join([f"-----BEGIN {label}-----", *lines, f"-----END {label}-----"]))
    return "\n".join(out) + "\n"


def write_secret_file(path: Path, content: str | bytes):
    """Записать секрет с правами 600 (без вывода содержимого)."""
    data = content.encode("utf-8") if isinstance(content, str) else content
    data = data.replace(b"\r\n", b"\n").strip() + b"\n"
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    with os.fdopen(fd, "wb") as fh:
        fh.write(data)
    os.chmod(path, 0o600)


def install_from_files(conn: KafkaConnection, ca: Path | str, cert: Path | str | None = None,
                       key: Path | str | None = None):
    """Скопировать скачанные из Aiven файлы в secrets_dir под стандартными именами."""
    prepare_secrets_dir(conn.secrets_dir)
    pairs = [(ca, conn.ca)]
    if conn.auth_mode == "certificate":
        pairs += [(cert, conn.cert), (key, conn.key)]
    for src, dst in pairs:
        if src is None or not Path(src).is_file():
            raise KafkaSecurityError(f"Не найден файл {dst.name}: {src}")
        if Path(src).resolve() != dst.resolve():
            write_secret_file(dst, Path(src).read_bytes())


# ---------------------------------------------------------------------------
# Проверка PEM и создание хранилищ
# ---------------------------------------------------------------------------
def _run(cmd: list[str], env_extra: dict | None = None, what: str = "") -> str:
    exe = cmd[0]
    if shutil.which(exe) is None:
        raise KafkaSecurityError(f"Не найдена утилита {exe}. Colab: входит в образ; Arch Linux: "
                                 f"{'openssl' if exe == 'openssl' else 'jdk21-openjdk'} (pacman -S)")
    env = dict(os.environ)
    env.update(env_extra or {})
    proc = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if proc.returncode != 0:
        # stderr OpenSSL/keytool не содержит ключей; обрезаем на всякий случай
        msg = (proc.stderr or proc.stdout).strip().splitlines()
        tail = " | ".join(msg[-3:])[:400]
        raise KafkaSecurityError(f"{what or exe} завершилась с ошибкой {proc.returncode}: {tail}")
    return proc.stdout


def _check_pem(path: Path, markers: tuple[str, ...], label: str):
    if not path.is_file() or path.stat().st_size == 0:
        raise KafkaSecurityError(f"{label}: файл {path.name} отсутствует или пуст")
    head = path.read_text(encoding="utf-8", errors="replace")
    if not any(f"-----BEGIN {m}-----" in head for m in markers):
        raise KafkaSecurityError(f"{label}: {path.name} не похож на PEM ({' / '.join(markers)})")


def validate_pem_files(conn: KafkaConnection) -> dict:
    """Проверить наличие, формат, срок действия и соответствие ключа сертификату.

    Возвращает только открытые сведения: subject, issuer, срок действия.
    """
    info = {}
    _check_pem(conn.ca, ("CERTIFICATE",), "CA certificate")
    info["ca"] = _cert_info(conn.ca)
    if conn.auth_mode == "certificate":
        _check_pem(conn.cert, ("CERTIFICATE",), "Service certificate")
        _check_pem(conn.key, ("PRIVATE KEY", "RSA PRIVATE KEY", "EC PRIVATE KEY"), "Service key")
        info["service_cert"] = _cert_info(conn.cert)
        cert_pub = _run(["openssl", "x509", "-in", str(conn.cert), "-noout", "-pubkey"], what="openssl x509 -pubkey")
        key_pub = _run(["openssl", "pkey", "-in", str(conn.key), "-pubout"], what="openssl pkey -pubout")
        if cert_pub.strip() != key_pub.strip():
            raise KafkaSecurityError("service.key не соответствует service.cert (разные открытые ключи)")
        info["key_matches_cert"] = True
        _run(["openssl", "verify", "-CAfile", str(conn.ca), str(conn.cert)], what="openssl verify")
        info["cert_signed_by_ca"] = True
    for k in ("ca", "service_cert"):
        if k in info and info[k]["expired"]:
            raise KafkaSecurityError(f"Сертификат {k} просрочен: {info[k]['not_after']}")
    return info


def _cert_info(path: Path) -> dict:
    out = _run(["openssl", "x509", "-in", str(path), "-noout", "-subject", "-issuer", "-enddate"],
               what=f"openssl x509 ({path.name})")
    fields = dict(line.split("=", 1) for line in out.strip().splitlines() if "=" in line)
    not_after = fields.get("notAfter", "").strip()
    expired = False
    try:
        dt = datetime.strptime(not_after, "%b %d %H:%M:%S %Y %Z").replace(tzinfo=timezone.utc)
        expired = dt < datetime.now(timezone.utc)
    except ValueError:
        pass
    return {"subject": fields.get("subject", "").strip(), "issuer": fields.get("issuer", "").strip(),
            "not_after": not_after, "expired": expired}


def build_java_stores(conn: KafkaConnection) -> dict:
    """Создать truststore и (для certificate) keystore в формате PKCS12 для JVM Spark.

    Повторный вызов пересоздаёт хранилища с текущим паролем сессии.
    """
    prepare_secrets_dir(conn.secrets_dir)
    env = {"LAB3_STORE_PASS": conn.store_password}
    for p in (conn.truststore, conn.keystore):
        if p.exists():
            p.unlink()
    _run(["keytool", "-importcert", "-noprompt", "-alias", "aiven-ca", "-file", str(conn.ca),
          "-keystore", str(conn.truststore), "-storetype", "PKCS12", "-storepass:env", "LAB3_STORE_PASS"],
         env, "keytool -importcert (truststore)")
    os.chmod(conn.truststore, 0o600)
    result = {"truststore": str(conn.truststore)}
    if conn.auth_mode == "certificate":
        _run(["openssl", "pkcs12", "-export", "-inkey", str(conn.key), "-in", str(conn.cert),
              "-certfile", str(conn.ca), "-name", "service_key", "-out", str(conn.keystore),
              "-passout", "env:LAB3_STORE_PASS"], env, "openssl pkcs12 -export (keystore)")
        os.chmod(conn.keystore, 0o600)
        listing = _run(["keytool", "-list", "-keystore", str(conn.keystore), "-storetype", "PKCS12",
                        "-storepass:env", "LAB3_STORE_PASS"], env, "keytool -list (keystore)")
        if "PrivateKeyEntry" not in listing:
            raise KafkaSecurityError("keystore создан, но не содержит PrivateKeyEntry")
        result["keystore"] = str(conn.keystore)
    listing = _run(["keytool", "-list", "-keystore", str(conn.truststore), "-storetype", "PKCS12",
                    "-storepass:env", "LAB3_STORE_PASS"], env, "keytool -list (truststore)")
    if "trustedCertEntry" not in listing:
        raise KafkaSecurityError("truststore создан, но не содержит trustedCertEntry")
    return result


def load_from_colab(conn: KafkaConnection, logger=None) -> str:
    """Получить PEM-файлы в Colab: из «Секретов» Colab, иначе через окно загрузки файлов.

    Секреты Colab (значок ключа слева): KAFKA_CA_PEM, KAFKA_SERVICE_CERT, KAFKA_SERVICE_KEY
    (содержимое файлов целиком). Если их нет — откроется диалог загрузки трёх файлов.
    """
    prepare_secrets_dir(conn.secrets_dir)
    needed = {CA_FILE: "KAFKA_CA_PEM"}
    if conn.auth_mode == "certificate":
        needed.update({CERT_FILE: "KAFKA_SERVICE_CERT", KEY_FILE: "KAFKA_SERVICE_KEY"})
    have = {name: (conn.secrets_dir / name).is_file() for name in needed}
    if all(have.values()):
        return "already-present"
    try:
        from google.colab import userdata  # type: ignore
        got = 0
        for name, secret_name in needed.items():
            try:
                value = userdata.get(secret_name)
            except Exception:
                value = None
            if value:
                write_secret_file(conn.secrets_dir / name, normalize_pem(value))
                got += 1
        if got == len(needed):
            return "colab-secrets"
    except ImportError:
        raise KafkaSecurityError("Не Colab: положите файлы в secrets_dir или используйте install_from_files()")
    from google.colab import files  # type: ignore
    print("Выберите файлы, скачанные из Aiven:", ", ".join(needed))
    uploaded = files.upload()
    for fname, content in uploaded.items():
        target = _match_uploaded(fname, list(needed))
        if target:
            write_secret_file(conn.secrets_dir / target, content)
        # загруженное также оказывается в текущем каталоге — удаляем копию
        try:
            os.remove(fname)
        except OSError:
            pass
    missing = [n for n in needed if not (conn.secrets_dir / n).is_file()]
    if missing:
        raise KafkaSecurityError(f"Не загружены файлы: {', '.join(missing)}")
    return "upload"


def _match_uploaded(fname: str, needed: list[str]) -> str | None:
    low = fname.lower()
    if "ca" in low and low.endswith(".pem"):
        return CA_FILE if CA_FILE in needed else None
    if low.endswith(".cert") or ("service" in low and low.endswith((".crt", ".pem"))):
        return CERT_FILE if CERT_FILE in needed else None
    if low.endswith(".key"):
        return KEY_FILE if KEY_FILE in needed else None
    return None

In [ ]:
%%writefile {CODE_DIR}/producer.py
#!/usr/bin/env python3
"""Kafka Producer для отправки новых банковских транзакций в Aiven (ЛР № 3).

Используется из блокнота Colab (функции) и как самостоятельный скрипт на Arch Linux:

  python producer.py check                         # подключение и наличие топика
  python producer.py send --file data/new_transactions.jsonl
  python producer.py read --max 5                  # прочитать последние сообщения с метаданными

Параметры подключения берутся из переменных окружения (файл .env, см. .env.example):
LAB3_BOOTSTRAP_SERVERS, LAB3_TOPIC, LAB3_AUTH_MODE, LAB3_SECRETS_DIR,
LAB3_SASL_USERNAME, LAB3_SASL_PASSWORD.

Каждая строка файла JSON Lines отправляется как value «как есть» (UTF-8),
поэтому битые строки тоже попадают в топик — их должен отбраковать Spark.
Ключ сообщения — transaction_id (если строку удалось разобрать).
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

from kafka_security import KafkaConnection, KafkaSecurityError


class KafkaUnavailableError(RuntimeError):
    """Брокер недоступен или топик не найден."""


def _confluent():
    try:
        import confluent_kafka
        import confluent_kafka.admin  # noqa: F401  (подмодуль AdminClient)
        return confluent_kafka
    except ImportError as exc:  # pragma: no cover - зависит от окружения
        raise RuntimeError("Не установлен confluent-kafka: pip install confluent-kafka==2.16.0") from exc


def check_connection(conf: dict, topic: str, timeout: float = 15.0) -> dict:
    """Запросить метаданные кластера. Возвращает число брокеров и партиций топика."""
    ck = _confluent()
    admin = ck.admin.AdminClient({**conf, "socket.connection.setup.timeout.ms": int(timeout * 1000),
                                  # реже переподключаться, чтобы не засорять вывод при недоступном брокере
                                  "reconnect.backoff.ms": 2000, "reconnect.backoff.max.ms": 10000})
    try:
        md = admin.list_topics(timeout=timeout)
    except ck.KafkaException as exc:
        raise KafkaUnavailableError(
            f"Kafka недоступна ({exc.args[0].name() if exc.args else exc}). Проверьте, что сервис Aiven "
            "в состоянии Running (бесплатный сервис выключается после простоя), адрес и порт, сертификаты.") from exc
    if topic not in md.topics:
        raise KafkaUnavailableError(f"Топик {topic!r} не найден. Создайте его в Aiven: Topics → Create topic. "
                                    f"Доступные топики: {sorted(t for t in md.topics if not t.startswith('__'))}")
    t = md.topics[topic]
    if t.error is not None:
        raise KafkaUnavailableError(f"Ошибка топика {topic}: {t.error}")
    return {"brokers": len(md.brokers), "topic": topic, "partitions": len(t.partitions),
            "replicas_per_partition": sorted({len(p.replicas) for p in t.partitions.values()})}


def read_lines(path: Path | str) -> list[str]:
    lines = [ln.rstrip("\n") for ln in Path(path).read_text(encoding="utf-8").splitlines()]
    return [ln for ln in lines if ln.strip()]


def _key_of(line: str) -> bytes | None:
    try:
        tid = json.loads(line).get("transaction_id")
        return str(tid).encode("utf-8") if tid else None
    except (json.JSONDecodeError, AttributeError):
        return None


def produce_lines(conf: dict, topic: str, lines: list[str], batch_label: str | None = None,
                  flush_timeout: float = 60.0, producer=None) -> list[dict]:
    """Отправить строки в топик, дождаться подтверждений и вернуть отчёт о доставке.

    Для каждого сообщения: ключ, партиция, offset, timestamp брокера, ошибка (если была).
    `producer` можно передать для тестов (объект с методами produce/poll/flush).
    """
    if not lines:
        return []
    batch_label = batch_label or f"send-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}-{uuid.uuid4().hex[:6]}"
    if producer is None:
        ck = _confluent()
        producer = ck.Producer({**conf, "acks": "all", "linger.ms": 20, "message.timeout.ms": 30000,
                                "compression.type": "gzip"})
    reports: list[dict] = []

    def on_delivery(err, msg):
        ts_type, ts = msg.timestamp() if msg is not None else (None, None)
        reports.append({
            "key": (msg.key() or b"").decode("utf-8", "replace") if msg is not None else None,
            "partition": msg.partition() if msg is not None else None,
            "offset": msg.offset() if msg is not None else None,
            "timestamp_ms": ts, "timestamp_utc": datetime.fromtimestamp(ts / 1000, timezone.utc).isoformat()
            if ts and ts > 0 else None,
            "error": str(err) if err else "",
        })

    for i, line in enumerate(lines):
        headers = [("source", b"lab3-producer"), ("schema_version", b"1"),
                   ("content_type", b"application/json"), ("send_batch", batch_label.encode()),
                   ("sequence", str(i).encode())]
        while True:
            try:
                producer.produce(topic, key=_key_of(line), value=line.encode("utf-8"),
                                 headers=headers, on_delivery=on_delivery)
                break
            except BufferError:  # локальная очередь заполнена — дать отправиться
                producer.poll(0.5)
        producer.poll(0)
    remaining = producer.flush(flush_timeout)
    if remaining:
        raise KafkaUnavailableError(f"Не подтверждено сообщений: {remaining} за {flush_timeout} с — брокер недоступен?")
    return reports


def read_messages(conf: dict, topic: str, start_offsets: dict[int, int] | None = None,
                  max_messages: int = 10, timeout: float = 20.0) -> list[dict]:
    """Прочитать сообщения с метаданными (partition, offset, timestamp, headers, key, value).

    start_offsets: {partition: offset}; если None — последние max_messages сообщений каждой партиции.
    Используется отдельная временная группа, offset'ы не фиксируются (не мешает Spark).
    """
    ck = _confluent()
    consumer = ck.Consumer({**conf, "group.id": f"lab3-verify-{uuid.uuid4().hex[:8]}",
                            "enable.auto.commit": False, "auto.offset.reset": "earliest"})
    try:
        md = consumer.list_topics(topic, timeout=timeout)
        if topic not in md.topics or md.topics[topic].error is not None:
            raise KafkaUnavailableError(f"Топик {topic} недоступен")
        parts = sorted(md.topics[topic].partitions)
        assignment = []
        for p in parts:
            low, high = consumer.get_watermark_offsets(ck.TopicPartition(topic, p), timeout=timeout)
            if start_offsets is not None:
                if p not in start_offsets:
                    continue
                off = start_offsets[p]
            else:
                off = max(low, high - max_messages)
            if off < high:
                assignment.append(ck.TopicPartition(topic, p, off))
        if not assignment:
            return []
        consumer.assign(assignment)
        out, deadline = [], time.time() + timeout
        while len(out) < max_messages * max(1, len(assignment)) and time.time() < deadline:
            msg = consumer.poll(1.0)
            if msg is None:
                if all(_at_end(consumer, ck, tp, timeout) for tp in consumer.assignment()):
                    break
                continue
            if msg.error():
                continue
            ts_type, ts = msg.timestamp()
            out.append({
                "partition": msg.partition(), "offset": msg.offset(),
                "timestamp_utc": datetime.fromtimestamp(ts / 1000, timezone.utc).isoformat() if ts > 0 else None,
                "key": (msg.key() or b"").decode("utf-8", "replace"),
                "headers": {k: (v or b"").decode("utf-8", "replace") for k, v in (msg.headers() or [])},
                "value": msg.value().decode("utf-8", "replace"),
            })
        return sorted(out, key=lambda r: (r["partition"], r["offset"]))
    finally:
        consumer.close()


def _at_end(consumer, ck, tp, timeout) -> bool:
    pos = consumer.position([tp])[0].offset
    _, high = consumer.get_watermark_offsets(ck.TopicPartition(tp.topic, tp.partition), timeout=timeout)
    return pos >= high


def summarize_delivery(reports: list[dict]) -> dict:
    ok = [r for r in reports if not r["error"]]
    by_part: dict[int, list[int]] = {}
    for r in ok:
        by_part.setdefault(r["partition"], []).append(r["offset"])
    return {"sent": len(reports), "delivered": len(ok), "failed": len(reports) - len(ok),
            "partitions": {p: {"min_offset": min(v), "max_offset": max(v), "count": len(v)}
                           for p, v in sorted(by_part.items())}}


# ---------------------------------------------------------------------------
# CLI для Arch Linux
# ---------------------------------------------------------------------------
def load_dotenv(path: str = ".env"):
    """Минимальная загрузка .env (KEY=VALUE), без перезаписи уже заданных переменных."""
    p = Path(path)
    if not p.is_file():
        return
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))


def connection_from_env() -> KafkaConnection:
    bootstrap = os.environ.get("LAB3_BOOTSTRAP_SERVERS", "")
    if not bootstrap:
        raise KafkaSecurityError("Не задана LAB3_BOOTSTRAP_SERVERS (скопируйте .env.example в .env)")
    return KafkaConnection(
        bootstrap_servers=bootstrap,
        auth_mode=os.environ.get("LAB3_AUTH_MODE", "certificate"),
        secrets_dir=Path(os.path.expanduser(os.environ.get("LAB3_SECRETS_DIR", "~/.config/lab3-kafka"))),
        sasl_username=os.environ.get("LAB3_SASL_USERNAME"),
        sasl_password=os.environ.get("LAB3_SASL_PASSWORD"),
    )


def main(argv=None):
    load_dotenv()
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("command", choices=["check", "send", "read"])
    p.add_argument("--topic", default=os.environ.get("LAB3_TOPIC", "bank-transactions-v2"))
    p.add_argument("--file", default="data/new_transactions.jsonl")
    p.add_argument("--max", type=int, default=5)
    p.add_argument("--report", default="", help="CSV-отчёт о доставке (для send)")
    a = p.parse_args(argv)
    try:
        conn = connection_from_env()
        conf = conn.confluent_config()
        if a.command == "check":
            print(json.dumps(check_connection(conf, a.topic), ensure_ascii=False, indent=2))
        elif a.command == "send":
            check_connection(conf, a.topic)
            reports = produce_lines(conf, a.topic, read_lines(a.file))
            print(json.dumps(summarize_delivery(reports), ensure_ascii=False, indent=2))
            if a.report:
                import lab3_common
                lab3_common.write_csv_atomic(a.report, reports)
        else:
            for m in read_messages(conf, a.topic, max_messages=a.max):
                print(json.dumps(m, ensure_ascii=False))
    except (KafkaSecurityError, KafkaUnavailableError) as exc:
        print(f"ОШИБКА: {exc}", file=sys.stderr)
        return 2
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile {CODE_DIR}/spark_jobs.py
"""PySpark: пакетная обработка CSV и Spark Structured Streaming из Kafka (ЛР № 3).

Ключевая идея: исходные данные (CSV и JSON из Kafka) читаются с ЯВНОЙ
схемой из строковых полей, после чего ОДНИ И ТЕ ЖЕ функции normalize() и
flag_rejects() безопасно приводят типы (try_cast / try_to_timestamp) и
назначают каждой некорректной записи одну причину отказа. Поэтому правила
очистки исторических и потоковых данных совпадают.

Модель «история + поток»:
  * очищенная история сохраняется в Parquet (clean_history.parquet);
  * принятые потоковые транзакции дописываются в устойчивое хранилище
    stream_transactions/batch_id=<N>/ внутри foreachBatch (повтор того же
    batch_id перезаписывает только его каталог — запись идемпотентна);
  * после каждого микропакета итоговые показатели ПЕРЕСЧИТЫВАЮТСЯ как
    агрегаты истории + агрегаты всех сохранённых потоковых транзакций.
Так результат не зависит от того, сколько раз перезапускался поток.
"""
from __future__ import annotations

import logging
import os
import shutil
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

import lab3_common as C

log = logging.getLogger("lab3")

RAW_SCHEMA = StructType([StructField(c, StringType(), True) for c in C.ALL_FIELDS])
JSON_SCHEMA = StructType([StructField(c, StringType(), True) for c in C.ALL_FIELDS]
                         + [StructField("_corrupt_record", StringType(), True)])
CATEGORICAL = ["transaction_type", "currency", "city", "device_type"]
NUMERIC = ["amount", "fee", "amount_eur"]
DIMENSIONS = ["city", "currency", "transaction_type", "device_type"]


# ===========================================================================
# SparkSession
# ===========================================================================
def kafka_package_coordinate(spark_version: str) -> str:
    """Координата Maven коннектора Kafka, совпадающая с версией Spark и Scala."""
    major = int(spark_version.split(".")[0])
    scala = "2.13" if major >= 4 else "2.12"
    return f"org.apache.spark:spark-sql-kafka-0-10_{scala}:{spark_version}"


def get_spark(settings: C.Settings, app_name: str = "lab3-bank-transactions", with_kafka: bool = True,
              master: str = "local[*]", driver_memory: str = "4g") -> SparkSession:
    """Создать SparkSession один раз; при повторном запуске ячейки вернуть существующую.

    Пакет Kafka можно подключить только до старта JVM: если активная сессия
    создана без него, нужно перезапустить среду выполнения (Runtime → Restart).
    """
    import pyspark

    active = SparkSession.getActiveSession()
    pkg = settings.kafka_package or kafka_package_coordinate(pyspark.__version__)
    if active is not None:
        loaded = active.sparkContext.getConf().get("spark.jars.packages", "")
        if with_kafka and "spark-sql-kafka" not in loaded:
            raise RuntimeError("Активная SparkSession создана без коннектора Kafka. "
                               "Выполните Runtime → Restart session и запустите ячейки заново.")
        log.info("Используется существующая SparkSession (%s)", active.sparkContext.applicationId)
        return active
    b = (SparkSession.builder.appName(app_name).master(master)
         .config("spark.driver.memory", driver_memory)
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", str(settings.shuffle_partitions))
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.jars.ivy", str(Path(settings.work_dir).parent / ".ivy2")))
    if with_kafka:
        b = b.config("spark.jars.packages", pkg)
    spark = b.getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    log.info("SparkSession: Spark %s, master %s, app %s, kafka package %s", spark.version,
             spark.sparkContext.master, spark.sparkContext.applicationId, pkg if with_kafka else "-")
    return spark


def environment_info(spark: SparkSession) -> dict:
    import platform
    import sys
    sc = spark.sparkContext
    jvm = sc._jvm.java.lang.System
    return {"spark_version": spark.version, "python": sys.version.split()[0], "platform": platform.platform(),
            "java_version": jvm.getProperty("java.version"), "java_vendor": jvm.getProperty("java.vendor"),
            "master": sc.master, "default_parallelism": sc.defaultParallelism, "app_id": sc.applicationId,
            "spark_ui": sc.uiWebUrl, "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
            "kafka_package": sc.getConf().get("spark.jars.packages", "")}


# ===========================================================================
# Общие правила нормализации и проверки
# ===========================================================================
def _blank_to_null(c):
    t = F.trim(F.col(c))
    return F.when(t == "", F.lit(None)).otherwise(t)


def _lit_map(d: dict, value_cast: str | None = None):
    items = []
    for k, v in d.items():
        val = F.lit(v).cast(value_cast) if value_cast else F.lit(v)
        items += [F.lit(k), val]
    return F.create_map(*items)


def normalize(df: DataFrame) -> DataFrame:
    """Строковые поля -> нормализованные и типизированные столбцы (исходные сохраняются)."""
    for c in C.ALL_FIELDS:
        df = df.withColumn(f"{c}_n", _blank_to_null(c))
    rates = _lit_map(C.EUR_RATES, "decimal(12,6)")
    cities = _lit_map({k.lower(): k for k in C.CITIES})
    ts_primary = F.try_to_timestamp(F.col("timestamp_n"), F.lit(C.SPARK_TIMESTAMP_FORMAT))
    ts_iso = F.try_to_timestamp(F.col("timestamp_n"), F.lit("yyyy-MM-dd'T'HH:mm:ss"))
    return (df
            .withColumn("amount_dec", F.expr("try_cast(amount_n AS DECIMAL(18,2))"))
            .withColumn("fee_dec", F.coalesce(F.expr("try_cast(fee_n AS DECIMAL(18,2))"), F.lit(0).cast("decimal(18,2)")))
            .withColumn("currency_norm", F.upper(F.col("currency_n")))
            .withColumn("rate_eur", F.try_element_at(rates, F.col("currency_norm")))
            .withColumn("city_norm", F.try_element_at(cities, F.lower(F.col("city_n"))))
            .withColumn("type_norm", F.lower(F.col("transaction_type_n")))
            .withColumn("device_norm", F.lower(F.col("device_type_n")))
            .withColumn("ts_parsed", F.coalesce(ts_primary, ts_iso))
            .withColumn("ts_primary_ok", ts_primary.isNotNull()))


def reject_rules(ts_min: str, ts_max: str) -> list[tuple[str, str]]:
    """Порядок важен: строка получает ПЕРВУЮ сработавшую причину (без двойного учёта)."""
    return [
        ("missing_transaction_id", "transaction_id_n IS NULL"),
        ("missing_client_id", "client_id_n IS NULL"),
        ("missing_amount", "amount_n IS NULL"),
        ("invalid_amount_format", "amount_dec IS NULL"),
        ("non_positive_amount", "amount_dec <= 0"),
        ("amount_too_large", f"amount_dec > {C.MAX_REASONABLE_AMOUNT}"),
        ("missing_timestamp", "timestamp_n IS NULL"),
        ("invalid_timestamp", "ts_parsed IS NULL"),
        ("timestamp_out_of_range",
         f"ts_parsed < TIMESTAMP '{ts_min} 00:00:00' OR ts_parsed >= TIMESTAMP '{ts_max} 00:00:00'"),
        ("missing_currency", "currency_n IS NULL"),
        ("unknown_currency", "rate_eur IS NULL"),
    ]


def flag_rejects(norm: DataFrame, ts_min: str, ts_max: str, extra_first: list | None = None) -> DataFrame:
    rules = (extra_first or []) + reject_rules(ts_min, ts_max)
    reason = F.lit(None).cast("string")
    for name, cond in reversed(rules):
        reason = F.when(F.expr(cond), F.lit(name)).otherwise(reason)
    return norm.withColumn("reject_reason", reason)


def to_clean(valid: DataFrame, keep: tuple[str, ...] = ()) -> DataFrame:
    """Итоговая типизированная схема; неизвестные категории -> 'unknown'.

    keep — дополнительные столбцы, которые нужно сохранить (например, метаданные Kafka).
    """
    known_types = list(C.TRANSACTION_TYPES)
    known_devices = list(C.DEVICE_TYPES)
    return valid.select(
        F.col("transaction_id_n").alias("transaction_id"),
        F.col("client_id_n").alias("client_id"),
        F.when(F.col("type_norm").isin(known_types), F.col("type_norm")).otherwise(F.lit(C.UNKNOWN)).alias("transaction_type"),
        F.col("amount_dec").alias("amount"),
        F.col("currency_norm").alias("currency"),
        F.coalesce(F.col("city_norm"), F.lit(C.UNKNOWN_CITY)).alias("city"),
        F.when(F.col("device_norm").isin(known_devices), F.col("device_norm")).otherwise(F.lit(C.UNKNOWN)).alias("device_type"),
        F.col("ts_parsed").alias("timestamp"),
        F.col("fee_dec").alias("fee"),
        (F.col("amount_dec") * F.col("rate_eur")).cast("decimal(18,2)").alias("amount_eur"),
        (F.col("fee_dec") * F.col("rate_eur")).cast("decimal(18,2)").alias("fee_eur"),
        *keep,
    )


def dedupe_by_id(valid: DataFrame) -> tuple[DataFrame, DataFrame]:
    """Оставить одну запись на transaction_id (детерминированно), вернуть (kept, removed)."""
    w = Window.partitionBy("transaction_id_n").orderBy(F.col("ts_primary_ok").desc(), F.xxhash64(*C.ALL_FIELDS))
    ranked = valid.withColumn("_rn", F.row_number().over(w))
    return ranked.filter("_rn = 1").drop("_rn"), ranked.filter("_rn > 1").drop("_rn")


# ===========================================================================
# Пакетная обработка
# ===========================================================================
def read_raw_csv(spark: SparkSession, path: str) -> DataFrame:
    return (spark.read.option("header", "true").option("mode", "PERMISSIVE").option("encoding", "UTF-8")
            .schema(RAW_SCHEMA).csv(path))


def quality_report(raw: DataFrame, settings: C.Settings) -> list[dict]:
    """Проверка качества исходных строк (до очистки)."""
    n = raw.count()
    norm = normalize(raw)
    c = F.col
    checks = []
    for col in C.ALL_FIELDS:
        checks.append((f"missing_{col}", col, f"{col}_n IS NULL", "NULL или пустая строка"))
    known_t, known_d = list(C.TRANSACTION_TYPES), list(C.DEVICE_TYPES)
    checks += [
        ("amount_not_numeric", "amount", "amount_n IS NOT NULL AND amount_dec IS NULL", "не число"),
        ("amount_non_positive", "amount", "amount_dec <= 0", "сумма <= 0"),
        ("amount_too_large", "amount", f"amount_dec > {C.MAX_REASONABLE_AMOUNT}", f"сумма > {C.MAX_REASONABLE_AMOUNT:,}"),
        ("timestamp_unparseable", "timestamp", "timestamp_n IS NOT NULL AND ts_parsed IS NULL", "не разбирается"),
        ("timestamp_iso_format", "timestamp", "ts_parsed IS NOT NULL AND NOT ts_primary_ok", "формат ISO с 'T' (исправимо)"),
        ("timestamp_out_of_range", "timestamp",
         f"ts_parsed < TIMESTAMP '{settings.period_start} 00:00:00' OR ts_parsed >= TIMESTAMP '{settings.period_end} 00:00:00'",
         f"вне [{settings.period_start}; {settings.period_end})"),
        ("currency_unknown", "currency", "currency_n IS NOT NULL AND rate_eur IS NULL", "нет в справочнике"),
        ("currency_wrong_case", "currency", "rate_eur IS NOT NULL AND currency <> currency_norm", "регистр (исправимо)"),
        ("city_unknown", "city", "city_n IS NOT NULL AND city_norm IS NULL", "нет в справочнике"),
        ("city_needs_normalization", "city", "city_norm IS NOT NULL AND city <> city_norm", "регистр/пробелы (исправимо)"),
        ("transaction_type_unknown", "transaction_type",
         f"type_norm IS NOT NULL AND type_norm NOT IN ({','.join(repr(t) for t in known_t)})", "нет в справочнике"),
        ("device_type_unknown", "device_type",
         f"device_norm IS NOT NULL AND device_norm NOT IN ({','.join(repr(t) for t in known_d)})", "нет в справочнике"),
        ("device_type_needs_normalization", "device_type",
         f"device_norm IN ({','.join(repr(t) for t in known_d)}) AND device_type <> device_norm", "регистр (исправимо)"),
    ]
    agg = norm.agg(*[F.sum(F.when(F.expr(cond), 1).otherwise(0)).alias(name) for name, _, cond, _ in checks]).first()
    rows = [{"check": name, "column": col, "rows": int(agg[name] or 0),
             "share_pct": round(100 * (agg[name] or 0) / n, 3) if n else 0, "description": desc}
            for name, col, _, desc in checks]
    distinct_rows = raw.dropDuplicates().count()
    dup_ids = (norm.filter(c("transaction_id_n").isNotNull()).groupBy("transaction_id_n").count()
               .filter("count > 1").agg(F.count(F.lit(1)).alias("ids"), F.sum("count").alias("rows")).first())
    rows += [
        {"check": "full_duplicate_rows", "column": "*", "rows": n - distinct_rows,
         "share_pct": round(100 * (n - distinct_rows) / n, 3) if n else 0, "description": "лишние копии полностью совпадающих строк"},
        {"check": "duplicated_transaction_ids", "column": "transaction_id", "rows": int(dup_ids["ids"] or 0),
         "share_pct": None, "description": f"id встречается >1 раза (строк с ними: {int(dup_ids['rows'] or 0)})"},
    ]
    return rows


def clean_batch(raw: DataFrame, settings: C.Settings) -> dict:
    """Очистка исторического набора. Возвращает DataFrame и сводку по шагам."""
    n_raw = raw.count()
    dedup = raw.dropDuplicates()                      # 1. полные дубликаты
    n_dedup = dedup.count()
    flagged = flag_rejects(normalize(dedup), settings.period_start, settings.period_end)
    reasons = {r["reject_reason"]: r["count"] for r in
               flagged.filter("reject_reason IS NOT NULL").groupBy("reject_reason").count().collect()}
    valid = flagged.filter("reject_reason IS NULL")
    kept, dup_ids = dedupe_by_id(valid)               # 3. частичные дубликаты по transaction_id
    clean = to_clean(kept)
    n_clean = clean.count()
    n_dup_ids = (n_dedup - sum(reasons.values())) - n_clean
    rejected = (flagged.filter("reject_reason IS NOT NULL").select(*C.ALL_FIELDS, "reject_reason")
                .unionByName(dup_ids.select(*C.ALL_FIELDS).withColumn("reject_reason", F.lit("duplicate_transaction_id"))))
    corrections = kept.agg(
        F.sum(F.when(F.col("city_n").isNull() | F.col("city_norm").isNull(), 1).otherwise(0)).alias("city_set_unknown"),
        F.sum(F.when(F.col("city_norm").isNotNull() & (F.col("city") != F.col("city_norm")), 1).otherwise(0)).alias("city_normalized"),
        F.sum(F.when(F.col("currency") != F.col("currency_norm"), 1).otherwise(0)).alias("currency_case_fixed"),
        F.sum(F.when(~F.col("ts_primary_ok"), 1).otherwise(0)).alias("timestamp_reformatted"),
        F.sum(F.when(F.col("type_norm").isNull() | ~F.col("type_norm").isin(list(C.TRANSACTION_TYPES)), 1).otherwise(0)).alias("type_set_unknown"),
        F.sum(F.when(F.col("device_norm").isNull() | ~F.col("device_norm").isin(list(C.DEVICE_TYPES)), 1).otherwise(0)).alias("device_set_unknown"),
    ).first().asDict()
    steps = [{"step": "0_raw_rows", "rows": n_raw, "description": "строк в исходном CSV (без заголовка)"},
             {"step": "1_full_duplicates_removed", "rows": n_raw - n_dedup, "description": "удалены полные дубликаты"}]
    for reason, cnt in sorted(reasons.items(), key=lambda kv: -kv[1]):
        steps.append({"step": f"2_rejected_{reason}", "rows": cnt, "description": "отклонено (одна причина на строку)"})
    steps += [{"step": "3_duplicate_transaction_id_removed", "rows": n_dup_ids,
               "description": "повтор transaction_id с другим оформлением"},
              {"step": "4_clean_rows", "rows": n_clean, "description": "строк после очистки"}]
    for k, v in corrections.items():
        steps.append({"step": f"fix_{k}", "rows": int(v or 0), "description": "исправлено в принятых строках (не удаление)"})
    balance_ok = n_raw == (n_raw - n_dedup) + sum(reasons.values()) + n_dup_ids + n_clean
    return {"clean": clean, "rejected": rejected, "steps": steps, "raw_rows": n_raw, "clean_rows": n_clean,
            "balance_ok": balance_ok}


def numeric_statistics(clean: DataFrame) -> list[dict]:
    out = []
    for col in NUMERIC:
        r = clean.agg(F.count(col).alias("n"), F.avg(col).alias("mean"), F.median(col).alias("median"),
                      F.stddev_samp(col).alias("std"), F.min(col).alias("min"), F.max(col).alias("max")).first()
        out.append({"column": col, "count": r["n"], "mean": _r(r["mean"]), "median": _r(r["median"]),
                    "stddev": _r(r["std"]), "min": _r(r["min"]), "max": _r(r["max"]),
                    "note": {"amount": "в исходной валюте операции (валюты смешаны — см. amount_eur)",
                             "fee": "комиссия в валюте операции",
                             "amount_eur": "сумма в EUR по условным курсам — сопоставимая величина"}[col]})
    return out


def _r(v, nd=2):
    return None if v is None else round(float(v), nd)


def top_categories(clean: DataFrame, n: int = 5) -> list[dict]:
    total = clean.count()
    rows = []
    for col in CATEGORICAL:
        for i, r in enumerate(clean.groupBy(col).count().orderBy(F.desc("count"), F.asc(col)).limit(n).collect(), 1):
            rows.append({"column": col, "rank": i, "value": r[col], "count": r["count"],
                         "share_pct": round(100 * r["count"] / total, 2)})
    return rows


def group_statistics(clean: DataFrame, by: list[str], amount_col: str = "amount_eur") -> list[dict]:
    g = (clean.groupBy(*by).agg(F.count(F.lit(1)).alias("tx_count"),
                                F.sum(amount_col).alias(f"{amount_col}_sum"),
                                F.avg(amount_col).alias(f"{amount_col}_avg"),
                                F.sum("fee_eur").alias("fee_eur_sum"))
         .orderBy(F.desc("tx_count"), *[F.asc(b) for b in by]))
    return [{**{b: r[b] for b in by}, "tx_count": r["tx_count"],
             f"{amount_col}_sum": _r(r[f"{amount_col}_sum"]), f"{amount_col}_avg": _r(r[f"{amount_col}_avg"]),
             "fee_eur_sum": _r(r["fee_eur_sum"])} for r in g.collect()]


def dimension_stats_df(clean: DataFrame) -> DataFrame:
    """Длинный формат (dimension, value, tx_count, amount_eur_sum) по всем измерениям."""
    parts = [clean.groupBy(F.lit(d).alias("dimension"), F.col(d).cast("string").alias("value"))
             .agg(F.count(F.lit(1)).alias("tx_count"), F.sum("amount_eur").alias("amount_eur_sum"))
             for d in DIMENSIONS]
    parts.append(clean.agg(F.lit("total").alias("dimension"), F.lit("all").alias("value"),
                           F.count(F.lit(1)).alias("tx_count"), F.sum("amount_eur").alias("amount_eur_sum")))
    out = parts[0]
    for p in parts[1:]:
        out = out.unionByName(p)
    return out


def rows_from_dimension_df(df: DataFrame) -> list[dict]:
    return [{"dimension": r["dimension"], "value": r["value"], "tx_count": int(r["tx_count"]),
             "amount_eur_sum": _r(r["amount_eur_sum"]),
             "avg_amount_eur": _r(r["amount_eur_sum"] / r["tx_count"]) if r["tx_count"] else None}
            for r in df.orderBy("dimension", F.desc("tx_count"), "value").collect()]


# ===========================================================================
# Потоковая обработка
# ===========================================================================
def kafka_stream(spark: SparkSession, kafka_options: dict, settings: C.Settings) -> DataFrame:
    return (spark.readStream.format("kafka").options(**kafka_options)
            .option("subscribe", settings.topic)
            .option("startingOffsets", settings.starting_offsets)
            .option("failOnDataLoss", "false")            # retention Aiven Free удаляет старые сообщения
            .option("maxOffsetsPerTrigger", str(settings.max_offsets_per_trigger))
            .option("includeHeaders", "true")
            .load())


def parse_messages(kdf: DataFrame) -> DataFrame:
    """value (байты) -> строка -> JSON по явной схеме; метаданные Kafka сохраняются."""
    value = F.col("value").cast("string")
    parsed = F.from_json(value, JSON_SCHEMA, {"mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt_record"})
    return (kdf.select(F.col("key").cast("string").alias("kafka_key"), value.alias("raw_value"),
                       F.col("topic"), F.col("partition"), F.col("offset"),
                       F.col("timestamp").alias("kafka_timestamp"), parsed.alias("j"))
            .select("kafka_key", "raw_value", "topic", "partition", "offset", "kafka_timestamp", "j.*")
            .withColumn("json_ok", F.col("_corrupt_record").isNull()
                        & F.col("raw_value").isNotNull() & (F.trim(F.col("raw_value")) != "")))


def validate_stream_batch(parsed: DataFrame, ts_min: str, ts_max: str) -> DataFrame:
    norm = normalize(parsed)
    return flag_rejects(norm, ts_min, ts_max, extra_first=[("invalid_json", "NOT json_ok")])


class StreamProcessor:
    """Обработчик foreachBatch: проверка, дедупликация, сохранение, пересчёт итогов."""

    def __init__(self, spark: SparkSession, settings: C.Settings, history_stats: DataFrame, history_ids: DataFrame,
                 ts_max: str | None = None):
        self.spark, self.s = spark, settings
        self.history_stats = history_stats      # агрегаты истории (малый DataFrame, кэширован)
        self.history_ids = history_ids          # transaction_id истории (кэширован)
        self.ts_max = ts_max or (datetime.now(timezone.utc) + timedelta(days=1)).strftime("%Y-%m-%d")
        self.batch_log = settings.results_dir / "stream_batches.csv"
        self.updated_csv = settings.results_dir / "updated_stream_statistics.csv"

    # ---- хранилище потоковых транзакций
    def stored_stream(self, exclude_batch: int | None = None) -> DataFrame | None:
        if not _has_parquet(self.s.stream_store_dir):
            return None
        df = self.spark.read.parquet(str(self.s.stream_store_dir))
        return df.filter(F.col("batch_id") != exclude_batch) if exclude_batch is not None else df

    def combined_stats(self) -> list[dict]:
        stats = self.history_stats
        stored = self.stored_stream()
        if stored is not None:
            stats = stats.unionByName(dimension_stats_df(stored))
        combined = stats.groupBy("dimension", "value").agg(F.sum("tx_count").alias("tx_count"),
                                                           F.sum("amount_eur_sum").alias("amount_eur_sum"))
        return rows_from_dimension_df(combined)

    def __call__(self, batch_df: DataFrame, batch_id: int):
        t0 = time.time()
        rows_in = batch_df.count()
        if rows_in == 0:
            log.info("batch %s: пустой микропакет — новых сообщений нет", batch_id)
            return
        flagged = validate_stream_batch(batch_df, self.s.new_period_start, self.ts_max).cache()
        valid = flagged.filter("reject_reason IS NULL")
        kept, dup_in_batch = dedupe_by_id(valid)
        # повтор уже обработанных ранее (в прошлых пакетах или в истории)
        seen = self.history_ids
        prev = self.stored_stream(exclude_batch=batch_id)
        if prev is not None:
            seen = seen.unionByName(prev.select("transaction_id"))
        kept_new = kept.join(seen.withColumnRenamed("transaction_id", "transaction_id_n"),
                             on="transaction_id_n", how="left_anti")
        dup_seen = kept.join(seen.withColumnRenamed("transaction_id", "transaction_id_n"),
                             on="transaction_id_n", how="left_semi")
        accepted = (to_clean(kept_new, keep=("partition", "offset", "kafka_timestamp"))
                    .withColumnRenamed("partition", "kafka_partition")
                    .withColumnRenamed("offset", "kafka_offset")
                    .withColumn("processed_at", F.current_timestamp()))
        n_acc = accepted.count()
        # Перезапись каталога ТОЛЬКО этого batch_id: повтор пакета после сбоя идемпотентен.
        out_dir = self.s.stream_store_dir / f"batch_id={batch_id}"
        if n_acc:
            accepted.write.mode("overwrite").parquet(str(out_dir))
        elif out_dir.exists():
            shutil.rmtree(out_dir)
        rejected = (flagged.filter("reject_reason IS NOT NULL")
                    .select("raw_value", "partition", "offset", "kafka_timestamp", "reject_reason")
                    .unionByName(dup_in_batch.select("raw_value", "partition", "offset", "kafka_timestamp")
                                 .withColumn("reject_reason", F.lit("duplicate_in_batch")))
                    .unionByName(dup_seen.select("raw_value", "partition", "offset", "kafka_timestamp")
                                 .withColumn("reject_reason", F.lit("duplicate_already_processed"))))
        reasons = {r["reject_reason"]: r["count"] for r in rejected.groupBy("reject_reason").count().collect()}
        rej_dir = self.s.stream_rejected_dir / f"batch_id={batch_id}"
        if reasons:
            rejected.write.mode("overwrite").parquet(str(rej_dir))
        flagged.unpersist()
        stats = self.combined_stats()
        C.write_csv_atomic(self.updated_csv, stats)
        entry = {"batch_id": batch_id, "processed_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                 "rows_in": rows_in, "accepted": n_acc, "rejected": sum(reasons.values()),
                 "rejected_by_reason": ";".join(f"{k}={v}" for k, v in sorted(reasons.items())),
                 "seconds": round(time.time() - t0, 2)}
        _upsert_batch_log(self.batch_log, entry)
        log.info("batch %s: получено %s, принято %s, отклонено %s %s", batch_id, rows_in, n_acc,
                 entry["rejected"], reasons or "")


def _has_parquet(path: Path) -> bool:
    """Есть ли ЗАФИКСИРОВАННЫЕ данные: маркер _SUCCESS в каталоге или в каталогах batch_id=N.

    Файлы из _temporary (запись ещё идёт) не считаются: Spark их игнорирует,
    и попытка прочитать такой каталог закончилась бы UNABLE_TO_INFER_SCHEMA.
    """
    if not path.is_dir():
        return False
    if (path / "_SUCCESS").exists():
        return True
    return any((d / "_SUCCESS").exists() for d in path.glob("batch_id=*"))


def _upsert_batch_log(path: Path, entry: dict):
    rows = [r for r in (C.read_csv(path) if path.exists() else []) if str(r["batch_id"]) != str(entry["batch_id"])]
    rows.append({k: str(v) for k, v in entry.items()})
    rows.sort(key=lambda r: int(r["batch_id"]))
    C.write_csv_atomic(path, rows, fieldnames=list(entry))


def start_streams(spark: SparkSession, kafka_options: dict, settings: C.Settings, processor: StreamProcessor) -> dict:
    """Запустить два независимых потоковых запроса, каждый со своим checkpoint.

    accepted_stream — foreachBatch: проверка, сохранение, пересчёт итоговых показателей.
    type_counts     — агрегат по типам операций только по потоку (outputMode complete, memory sink).
    Уже запущенные запросы с теми же именами сначала корректно останавливаются.
    """
    stop_streams(spark)
    parsed = parse_messages(kafka_stream(spark, kafka_options, settings))
    q_main = (parsed.writeStream.queryName("accepted_stream")
              .foreachBatch(processor)
              .option("checkpointLocation", str(settings.checkpoint_dir / "accepted_stream"))
              .trigger(processingTime=f"{settings.trigger_seconds} seconds")
              .start())
    # Второй поток: потоковая агрегация с состоянием. dropDuplicates по transaction_id
    # хранит множество увиденных id в состоянии (для учебного объёма это приемлемо).
    valid = (validate_stream_batch(parse_messages(kafka_stream(spark, kafka_options, settings)),
                                   settings.new_period_start, processor.ts_max)
             .filter("reject_reason IS NULL").transform(to_clean).dropDuplicates(["transaction_id"]))
    q_types = (valid.groupBy("transaction_type")
               .agg(F.count(F.lit(1)).alias("tx_count"), F.sum("amount_eur").alias("amount_eur_sum"))
               .writeStream.queryName("stream_type_counts").outputMode("complete").format("memory")
               .option("checkpointLocation", str(settings.checkpoint_dir / "stream_type_counts"))
               .trigger(processingTime=f"{settings.trigger_seconds} seconds")
               .start())
    log.info("Запущены потоки: accepted_stream (%s), stream_type_counts (%s); startingOffsets=%s, trigger=%ss",
             q_main.id, q_types.id, settings.starting_offsets, settings.trigger_seconds)
    return {"accepted_stream": q_main, "stream_type_counts": q_types}


def stop_streams(spark: SparkSession, timeout_s: int = 60):
    for q in list(spark.streams.active):
        if q.name in ("accepted_stream", "stream_type_counts"):
            log.info("Остановка потока %s (%s)", q.name, q.id)
            q.stop()
            try:
                q.awaitTermination(timeout_s)
            except Exception as exc:  # остановленный запрос может вернуть исключение прошлого сбоя
                log.warning("Поток %s завершился с ошибкой: %s", q.name, exc)


def find_stored_ids(settings: C.Settings, spark: SparkSession, ids: set[str]) -> set[str]:
    """Какие из указанных transaction_id уже есть в хранилище потоковых транзакций."""
    if not ids or not _has_parquet(settings.stream_store_dir):
        return set()
    try:
        df = spark.read.parquet(str(settings.stream_store_dir)).select("transaction_id")
        return {r[0] for r in df.filter(F.col("transaction_id").isin(list(ids))).collect()}
    except Exception as exc:  # каталог как раз перезаписывается микропакетом — повторим при следующем опросе
        log.debug("Хранилище потока пока недоступно для чтения: %s", exc)
        return set()


def wait_for_transactions(settings: C.Settings, spark: SparkSession, expected_ids: set[str],
                          query=None, timeout_s: int = 180, poll_s: float = 3.0) -> dict:
    """Ждать, пока ожидаемые transaction_id появятся в хранилище потоковых транзакций."""
    deadline = time.time() + timeout_s
    found = set()
    while time.time() < deadline:
        if query is not None and query.exception() is not None:
            raise RuntimeError(f"Поток упал: {query.exception()}")
        found = find_stored_ids(settings, spark, expected_ids)
        if found >= expected_ids:
            break
        time.sleep(poll_s)
    # данные уже записаны, но foreachBatch ещё пересчитывает итоги — дождаться конца микропакета
    while query is not None and query.status.get("isTriggerActive") and time.time() < deadline:
        time.sleep(0.5)
    return {"expected": len(expected_ids), "found": len(found), "missing": sorted(expected_ids - found)[:10],
            "complete": found >= expected_ids}


def wait_for_new_input(query, after_batch_id: int, min_rows: int, timeout_s: int = 120, poll_s: float = 2.0) -> dict:
    """Ждать, пока запрос обработает хотя бы min_rows строк в микропакетах после after_batch_id."""
    deadline = time.time() + timeout_s
    rows, batches = 0, []
    while time.time() < deadline:
        if query.exception() is not None:
            raise RuntimeError(f"Поток упал: {query.exception()}")
        progress = [p for p in (query.recentProgress or []) if p.get("batchId", -1) > after_batch_id]
        rows = sum(int(p.get("numInputRows", 0)) for p in progress)
        batches = [p["batchId"] for p in progress if int(p.get("numInputRows", 0)) > 0]
        if rows >= min_rows and not query.status.get("isTriggerActive", False):
            break
        time.sleep(poll_s)
    return {"rows_processed": rows, "batches_with_data": batches, "complete": rows >= min_rows}


def last_batch_id(query) -> int:
    lp = query.lastProgress
    return int(lp["batchId"]) if lp else -1


# ===========================================================================
# Резервная копия рабочего каталога в Google Drive
# ===========================================================================
def backup_work_dir(settings: C.Settings) -> Path:
    """Скопировать checkpoint и хранилище потока в Drive (после остановки потоков)."""
    dst = settings.work_backup_dir
    tmp = dst.with_name(dst.name + ".tmp")
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True)
    for name in ("checkpoints", "stream_transactions", "stream_rejected"):
        src = Path(settings.work_dir) / name
        if src.exists():
            shutil.copytree(src, tmp / name)
    if dst.exists():
        shutil.rmtree(dst)
    os.replace(tmp, dst)
    return dst


def restore_work_dir(settings: C.Settings) -> bool:
    """Восстановить рабочий каталог из Drive, если локальная копия отсутствует (новая ВМ Colab)."""
    src = settings.work_backup_dir
    local_ckpt = Path(settings.work_dir) / "checkpoints"
    if not src.exists() or (local_ckpt.exists() and any(local_ckpt.iterdir())):
        return False
    for name in ("checkpoints", "stream_transactions", "stream_rejected"):
        if (src / name).exists():
            target = Path(settings.work_dir) / name
            if target.exists():
                shutil.rmtree(target)
            shutil.copytree(src / name, target)
    return True

In [ ]:
# Импорт модулей (reload — чтобы повторный запуск подхватил изменения)
import lab3_common as C, generate_data as G, kafka_security as K, producer as P, spark_jobs as J
for _m in (C, G, K, P, J):
    importlib.reload(_m)
print("Модули загружены из", CODE_DIR)

## 2. Настройки (все параметры — в одном месте)

In [ ]:
settings = C.Settings(
    base_dir="/content/drive/MyDrive/lab3_bank" if IN_COLAB else os.environ["LAB3_BASE_DIR"],
    work_dir="/content/lab3_work" if IN_COLAB else os.environ.get("LAB3_WORK_DIR", "/tmp/lab3_work"),
    secrets_dir="/content/kafka_secrets" if IN_COLAB else os.environ.get("LAB3_SECRETS_DIR", "/tmp/lab3_secrets"),
    seed=42, raw_rows=500_000,                 # основной набор (~500 000 строк)
    new_rows=300, new_seed=2026, batch_tag="B001",  # пакет новых транзакций для Kafka
    topic="bank-transactions-v2",
    auth_mode="certificate",                   # "certificate" (ca.pem + service.cert + service.key) или "sasl"
    trigger_seconds=5, starting_offsets="earliest", shuffle_partitions=4,
)
REGENERATE_RAW = False          # True — пересоздать raw_data.csv (иначе используется существующий)
CREATE_TOPIC_IF_MISSING = os.environ.get("LAB3_CREATE_TOPIC", "0") == "1"  # в Aiven топик создаётся в UI
RUN_RESEND_DEMO = True          # повторная отправка того же пакета (проверка идемпотентности)
print({k: v for k, v in settings.as_dict().items()})

## 3. Google Drive и журнал выполнения

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
settings.ensure_dirs()
log = C.setup_logging(settings.results_dir / "execution_log.txt")
log.info("==== Запуск блокнота ЛР3; среда=%s; base_dir=%s", "colab" if IN_COLAB else "local", settings.base_dir)
for p in (settings.data_dir, settings.results_dir, settings.intermediate_dir, settings.work_dir):
    print(p)

## 4. Генерация синтетических данных

In [ ]:
import json
if settings.raw_csv.exists() and not REGENERATE_RAW:
    raw_stats = json.load(open(settings.raw_csv.with_suffix(".stats.json"), encoding="utf-8")) \
        if settings.raw_csv.with_suffix(".stats.json").exists() else {"file": str(settings.raw_csv)}
    log.info("raw_data.csv уже существует — генерация пропущена (REGENERATE_RAW=False)")
else:
    raw_stats = G.generate_raw(settings.raw_csv, settings.raw_rows, settings.seed,
                               settings.period_start, settings.period_end)
    log.info("Сгенерирован raw_data.csv: %s строк, %.1f МБ", raw_stats["total_rows_in_file"], raw_stats["file_size_mb"])
new_stats = G.generate_new(settings.new_jsonl, settings.new_rows, settings.new_seed, settings.new_period_start,
                           settings.batch_tag, out_csv=settings.new_csv)
log.info("Пакет для Kafka: %s сообщений (%s корректных)", new_stats["total_messages"], new_stats["valid_transactions"])
print(json.dumps(raw_stats, ensure_ascii=False, indent=1))
print(json.dumps(new_stats, ensure_ascii=False, indent=1))
with open(settings.raw_csv, encoding="utf-8") as fh:
    for _ in range(4):
        print(fh.readline().rstrip())

## 5. SparkSession

In [ ]:
spark = J.get_spark(settings)          # повторный запуск вернёт существующую сессию
env_info = J.environment_info(spark)
C.write_json(settings.results_dir / "environment.json", env_info)
for k, v in env_info.items():
    print(f"{k:20}: {v}")

## 6. Пакетная обработка исторических данных

CSV читается с **явной схемой**: все поля — строки (в данных есть «грязные» значения). Типы приводятся безопасно: `try_cast` для сумм, `try_to_timestamp` для дат. Ошибка преобразования даёт NULL и причину отказа, а не падение задания (в Spark 4 по умолчанию включён режим ANSI).

In [ ]:
raw = J.read_raw_csv(spark, str(settings.raw_csv)).cache()
n_rows, n_cols = raw.count(), len(raw.columns)
raw.printSchema()
raw.show(5, truncate=False)
print(f"Строк: {n_rows}, столбцов: {n_cols}")
log.info("Прочитан raw_data.csv: %s строк x %s столбцов", n_rows, n_cols)

In [ ]:
quality = J.quality_report(raw, settings)
C.write_csv_atomic(settings.results_dir / "data_quality_report.csv", quality)
C.show_rows(quality, 40, "Отчёт о качестве исходных данных (data_quality_report.csv)")

In [ ]:
res = J.clean_batch(raw, settings)
clean = res["clean"].cache()
C.write_csv_atomic(settings.results_dir / "cleaning_summary.csv", res["steps"])
C.show_rows(res["steps"], 40, "Очистка (cleaning_summary.csv)")
print(f"До очистки: {res['raw_rows']}, после: {res['clean_rows']}, баланс сходится: {res['balance_ok']}")
assert res["balance_ok"], "сумма удалённых и оставшихся строк не равна исходной"
sample = [r.asDict() for r in res["rejected"].orderBy("reject_reason").limit(50).collect()]
C.write_csv_atomic(settings.results_dir / "rejected_sample.csv", sample)
log.info("Очистка: %s -> %s строк", res["raw_rows"], res["clean_rows"])

In [ ]:
from pyspark.sql import functions as F
num = J.numeric_statistics(clean)
C.write_csv_atomic(settings.results_dir / "numeric_statistics.csv", num)
C.show_rows(num, title="Числовая статистика (numeric_statistics.csv)")
print("Исключены из числовой статистики: transaction_id, client_id — идентификаторы (среднее бессмысленно); "
      "timestamp — время (см. min/max ниже).")
clean.agg(F.min("timestamp").alias("min_timestamp"), F.max("timestamp").alias("max_timestamp")).show()

In [ ]:
top = J.top_categories(clean, 5)
C.write_csv_atomic(settings.results_dir / "top_categories.csv", top)
C.show_rows(top, 25, "Топ-5 значений категориальных столбцов (top_categories.csv)")

In [ ]:
groups = {
    "city_statistics.csv": J.group_statistics(clean, ["city"]),
    "currency_statistics.csv": J.group_statistics(clean, ["currency"], amount_col="amount"),
    "transaction_type_statistics.csv": J.group_statistics(clean, ["transaction_type"]),
    "city_currency_type_statistics.csv": J.group_statistics(clean, ["city", "currency", "transaction_type"]),
}
for name, rows in groups.items():
    C.write_csv_atomic(settings.results_dir / name, rows)
C.show_rows(groups["city_statistics.csv"], 15, "Транзакции по городам (city_statistics.csv; суммы в EUR по условным курсам)")
C.show_rows(groups["currency_statistics.csv"], 10, "По валютам (суммы в исходной валюте)")
C.show_rows(groups["transaction_type_statistics.csv"], 10, "По типам операций")

In [ ]:
# Очищенная история: Parquet на локальном диске (для Spark) + копия в Drive (промежуточные данные)
clean.write.mode("overwrite").parquet(str(settings.clean_history_dir))
drive_copy = settings.intermediate_dir / "clean_history.parquet"
if drive_copy.exists():
    shutil.rmtree(drive_copy)
shutil.copytree(settings.clean_history_dir, drive_copy)
history = spark.read.parquet(str(settings.clean_history_dir))
history_stats = J.dimension_stats_df(history).cache()
history_ids = history.select("transaction_id").cache()
history_stats.count(); history_ids.count()
C.write_csv_atomic(settings.results_dir / "history_dimension_statistics.csv", J.rows_from_dimension_df(history_stats))
log.info("История сохранена: %s; агрегаты по измерениям: %s строк", drive_copy, history_stats.count())

## 7. Подключение к Kafka в Aiven (SSL)

* Python-продюсер (librdkafka) использует PEM-файлы напрямую.
* Spark Kafka connector работает в JVM, поэтому из PEM создаются `client.truststore.p12` (CA) и `client.keystore.p12` (сертификат + ключ). Пароли хранилищ генерируются случайно на сессию.
* Секреты лежат только в `/content/kafka_secrets` на диске ВМ (права 700/600) и в Google Drive не попадают. Содержимое ключа и пароли не выводятся.

In [ ]:
def _secret(name, prompt=None, hidden=False):
    if IN_COLAB:
        from google.colab import userdata
        try:
            v = userdata.get(name)
            if v:
                return v.strip()
        except Exception:
            pass
        if prompt:
            import getpass
            return (getpass.getpass(prompt) if hidden else input(prompt)).strip()
        return None
    return os.environ.get("LAB3_" + name.replace("KAFKA_", ""), "").strip() or None

bootstrap = _secret("KAFKA_BOOTSTRAP_SERVERS", "Service URI Aiven (host:port): ")
conn = K.KafkaConnection(bootstrap_servers=bootstrap, auth_mode=settings.auth_mode,
                         secrets_dir=settings.secrets_dir,
                         sasl_username=_secret("KAFKA_SASL_USERNAME", "SASL username: ") if settings.auth_mode == "sasl" else None,
                         sasl_password=_secret("KAFKA_SASL_PASSWORD", "SASL password: ", hidden=True) if settings.auth_mode == "sasl" else None)
if IN_COLAB:
    source = K.load_from_colab(conn)
else:
    K.install_from_files(conn, os.environ["LAB3_CA_FILE"], os.environ.get("LAB3_CERT_FILE"), os.environ.get("LAB3_KEY_FILE"))
    source = "local-files"
cert_info = K.validate_pem_files(conn)
stores = K.build_java_stores(conn)
spark_kafka_options = conn.spark_options()     # содержит пароли — не печатать
producer_conf = conn.confluent_config()
print("Источник сертификатов:", source)
print("Подключение:", conn.describe())
print("Сертификаты (открытые сведения):", json.dumps(cert_info, ensure_ascii=False, indent=1))
print("Созданы хранилища:", [os.path.basename(v) for v in stores.values()])
log.info("SSL настроен: %s", conn.describe())

In [ ]:
# Проверка подключения и топика
if CREATE_TOPIC_IF_MISSING:     # только для локального стенда; в Aiven топик создаётся в веб-интерфейсе
    from confluent_kafka.admin import AdminClient, NewTopic
    _admin = AdminClient(producer_conf)
    if settings.topic not in _admin.list_topics(timeout=15).topics:
        _admin.create_topics([NewTopic(settings.topic, 2, 1)])[settings.topic].result()
topic_info = P.check_connection(producer_conf, settings.topic)
print(topic_info)
log.info("Kafka доступна: %s", topic_info)

In [ ]:
# Граничный случай: недоступный брокер — понятная ошибка вместо зависания
try:
    P.check_connection({**producer_conf, "bootstrap.servers": "localhost:1"}, settings.topic, timeout=5)
except P.KafkaUnavailableError as exc:
    print("Ожидаемая ошибка обработана:", str(exc)[:160])

## 8. Spark Structured Streaming

* Kafka **хранит** сообщения в топике (retention), а Spark читает их **микропакетами**: раз в `trigger` секунд он берёт offset'ы, появившиеся с прошлого пакета, и обрабатывает их как небольшой DataFrame.
* `startingOffsets="earliest"` действует только при **первом** запуске с пустым checkpoint. Дальше Spark продолжает с offset'ов, записанных в checkpoint. Повторная обработка уже прочитанных сообщений безопасна: дубликаты по `transaction_id` отбрасываются.
* Запрос `accepted_stream` (`foreachBatch`) проверяет сообщения, дописывает принятые транзакции в устойчивое хранилище Parquet и **пересчитывает** итоговые показатели «история + поток».
* Запрос `stream_type_counts` — потоковая агрегация с состоянием (`groupBy` + `outputMode("complete")`, memory sink) только по потоку.
* У каждого запроса свой checkpoint. Checkpoint не удаляется при повторном запуске.

In [ ]:
restored = J.restore_work_dir(settings)
print("Рабочий каталог восстановлен из Drive:", restored)
processor = J.StreamProcessor(spark, settings, history_stats, history_ids)
initial_stats = processor.combined_stats()
C.write_csv_atomic(settings.results_dir / "initial_stream_statistics.csv", initial_stats)
C.show_rows([r for r in initial_stats if r["dimension"] in ("total", "transaction_type")], 10,
            "Показатели ДО новых сообщений (initial_stream_statistics.csv)")

In [ ]:
import time
queries = J.start_streams(spark, spark_kafka_options, settings, processor)
q_main, q_types = queries["accepted_stream"], queries["stream_type_counts"]
time.sleep(settings.trigger_seconds * 3)
print("Статус accepted_stream:", q_main.status)
print("Последний прогресс:", {k: q_main.lastProgress.get(k) for k in ("batchId", "numInputRows")} if q_main.lastProgress else None)
print("Если новых сообщений нет, Spark ждёт: isDataAvailable=False, микропакеты с данными не создаются.")

In [ ]:
# Отправка новых транзакций в Kafka (Producer)
expected_ids = {f"S-{settings.batch_tag}-{i:06d}" for i in range(1, settings.new_rows + 1)}
already_ids = J.find_stored_ids(settings, spark, expected_ids)
expected_delta = len(expected_ids - already_ids)
if already_ids:
    print(f"ВНИМАНИЕ: {len(already_ids)} транзакций пакета {settings.batch_tag} уже обработаны ранее — они будут "
          "отклонены как повторы. Для новой порции измените batch_tag в разделе 2.")
lines = P.read_lines(settings.new_jsonl)
reports = P.produce_lines(producer_conf, settings.topic, lines, batch_label=f"lab3-{settings.batch_tag}")
C.write_csv_atomic(settings.results_dir / "kafka_delivery_report.csv", reports)
delivery = P.summarize_delivery(reports)
print(json.dumps(delivery, ensure_ascii=False, indent=1))
log.info("Отправлено в Kafka: %s", delivery)
assert delivery["failed"] == 0, "часть сообщений не доставлена"

In [ ]:
# Независимая проверка доставки: читаем отправленные сообщения обратно с метаданными
start_offsets = {p: v["min_offset"] for p, v in delivery["partitions"].items()}
msgs = P.read_messages(producer_conf, settings.topic, start_offsets, max_messages=len(lines))
C.write_csv_atomic(settings.results_dir / "kafka_messages_sample.csv",
                   [{**m, "headers": json.dumps(m["headers"])} for m in msgs])
C.show_rows([{**m, "value": m["value"][:80]} for m in msgs], 6, "Сообщения в топике: partition / offset / timestamp / headers / key / value")
print("Прочитано из топика:", len(msgs), "из", len(lines))

In [ ]:
# Ожидание обработки микропакетами Spark
wait = J.wait_for_transactions(settings, spark, expected_ids, query=q_main, timeout_s=240)
print(wait)
J.wait_for_new_input(q_types, -1, 1, timeout_s=60)
C.show_rows(C.read_csv(settings.results_dir / "stream_batches.csv"), 20, "Журнал микропакетов (stream_batches.csv)")
assert wait["complete"], "не все новые транзакции обработаны Spark — см. журнал и статус потока"

In [ ]:
print("Потоковая агрегация с состоянием (memory sink, outputMode complete):")
spark.sql("SELECT * FROM stream_type_counts ORDER BY tx_count DESC").show(truncate=False)
rej = spark.read.parquet(str(settings.stream_rejected_dir)) if J._has_parquet(settings.stream_rejected_dir) else None
rej_summary = []
if rej is not None:
    rej_summary = [r.asDict() for r in rej.groupBy("reject_reason").count().orderBy("reject_reason").collect()]
    C.write_csv_atomic(settings.results_dir / "stream_rejected_summary.csv", rej_summary)
    C.show_rows(rej_summary, title="Отклонённые сообщения потока (битый JSON, нет поля, повтор, неверные значения)")

In [ ]:
# Сравнение показателей до и после
updated_stats = C.read_csv(settings.results_dir / "updated_stream_statistics.csv")
comparison = C.compare_dimension_stats(initial_stats, updated_stats)
C.write_csv_atomic(settings.results_dir / "analysis_comparison.csv", comparison)
changed = [r for r in comparison if r["tx_count_delta"] != 0]
C.show_rows([r for r in changed if r["dimension"] in ("total", "transaction_type", "currency")], 15,
            "Изменения после новых сообщений (analysis_comparison.csv)")
total = next(r for r in comparison if r["dimension"] == "total")
print(f"Всего транзакций: {total['tx_count_before']} -> {total['tx_count_after']} (+{total['tx_count_delta']}; "
      f"ожидалось +{expected_delta})")
log.info("Итоги обновлены: total %s -> %s", total["tx_count_before"], total["tx_count_after"])

In [ ]:
# Повторная отправка того же пакета: все сообщения — повторы, показатели не должны измениться
if RUN_RESEND_DEMO:
    before = {(r["dimension"], r["value"]): r["tx_count"] for r in C.read_csv(settings.results_dir / "updated_stream_statistics.csv")}
    b0 = J.last_batch_id(q_main)
    P.produce_lines(producer_conf, settings.topic, lines, batch_label=f"lab3-{settings.batch_tag}-resend")
    print(J.wait_for_new_input(q_main, b0, len(lines), timeout_s=180))
    after = {(r["dimension"], r["value"]): r["tx_count"] for r in C.read_csv(settings.results_dir / "updated_stream_statistics.csv")}
    print("Показатели после повторной отправки не изменились:", before == after)
    C.show_rows(C.read_csv(settings.results_dir / "stream_batches.csv")[-3:], title="Последние микропакеты")

In [ ]:
# Корректная остановка потоков и резервная копия checkpoint/хранилища в Google Drive
J.stop_streams(spark)
print("Активные потоки:", [q.name for q in spark.streams.active])
backup = J.backup_work_dir(settings)
log.info("Потоки остановлены; рабочий каталог скопирован в %s", backup)
print("Резервная копия:", backup)

## 9. Результаты в Google Drive

In [ ]:
for d in (settings.data_dir, settings.results_dir):
    print(f"\n{d}")
    for p in sorted(d.iterdir()):
        if p.is_file():
            print(f"  {p.name:42} {p.stat().st_size / 1024:10.1f} КБ")
print("\nПромежуточные данные:", sorted(p.name for p in settings.intermediate_dir.iterdir()))

## 10. Самопроверка

In [ ]:
checks = []
def check(name, ok, detail=""):
    checks.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})

check("CSV содержит все обязательные поля", set(C.REQUIRED_FIELDS) <= set(raw.columns), ",".join(raw.columns))
check("масштаб данных ~500 000 строк", 400_000 <= n_rows <= 600_000, str(n_rows))
qmap = {r["check"]: r["rows"] for r in quality}
check("обнаружены пропуски", sum(v for k, v in qmap.items() if k.startswith("missing_")) > 0)
check("обнаружены дубликаты", qmap["full_duplicate_rows"] > 0, str(qmap["full_duplicate_rows"]))
check("очистка: баланс строк сходится", res["balance_ok"], f"{res['raw_rows']} -> {res['clean_rows']}")
check("SparkSession создана", spark.version.startswith("4."), spark.version)
city_sum = sum(r["tx_count"] for r in groups["city_statistics.csv"])
check("сумма по городам = числу очищенных строк", city_sum == res["clean_rows"], str(city_sum))
check("подключение к Kafka", topic_info["partitions"] >= 1, json.dumps(topic_info))
check("сообщения доставлены", delivery["failed"] == 0 and delivery["delivered"] == len(lines), str(delivery["delivered"]))
check("сообщения прочитаны из топика", len(msgs) == len(lines), str(len(msgs)))
check("Spark обработал новые транзакции", wait["complete"], f"{wait['found']}/{wait['expected']}")
check("итоги изменились на число новых принятых", total["tx_count_delta"] == expected_delta,
      f"{total['tx_count_delta']} (ожидалось {expected_delta})")
check("битый JSON отклонён", any(r["reject_reason"] == "invalid_json" for r in rej_summary))
check("сообщение без обязательного поля отклонено", any(r["reject_reason"] == "missing_client_id" for r in rej_summary))
check("повторы отклонены", any(r["reject_reason"].startswith("duplicate") for r in rej_summary))
check("результаты сохранены в Drive", all((settings.results_dir / f).exists() for f in [
    "data_quality_report.csv", "numeric_statistics.csv", "top_categories.csv", "city_statistics.csv",
    "initial_stream_statistics.csv", "updated_stream_statistics.csv", "analysis_comparison.csv", "execution_log.txt"]))
C.write_csv_atomic(settings.results_dir / "notebook_checks.csv", checks)
C.show_rows(checks, 30)
log.info("Самопроверка: %s PASS из %s", sum(c["result"] == "PASS" for c in checks), len(checks))

## 11. Повторный запуск и остановка

* **Ещё одна порция данных:** измените `batch_tag` (например, `B002`) в разделе 2 и выполните ячейки 4 и 8.
* **Новая сессия Colab:** выполните все ячейки. Checkpoint и хранилище потока восстановятся из `lab3_intermediate/work_backup`, и Spark продолжит с сохранённых offset'ов.
* **Начать поток «с нуля»:** удалите `lab3_intermediate/work_backup` и `/content/lab3_work`. Тогда при `startingOffsets="earliest"` Spark заново прочитает топик (в пределах retention), а дубликаты по `transaction_id` не исказят итоги.
* `startingOffsets="latest"` при пустом checkpoint пропустит сообщения, отправленные до запуска потока.
* Остановка в любой момент: `J.stop_streams(spark)`, затем `spark.stop()`.